In [1]:
%pip install --quiet pandas pyarrow geopandas scikit-learn matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Análises e diagnósticos — caminho para 100%

Lê o output final do `1_pipeline_etl_sp.ipynb` e:

1. **Diagnostica** a cobertura final e quem ainda falta (2,75% legítimos sem renda).
2. **Testa caminhos para chegar perto de 100%**:
    - KNN espacial (mediana dos K vizinhos geográficos mais próximos)
    - Modelo preditivo simples (regressor com atributos do setor)
3. **Avalia honestamente** se vale ir além de 97,25% — ou se os ~2,75% são essencialmente "sem residentes para ter renda".

**Nota**: o objetivo não é fingir 100% por força — é entender **se** é possível e **com que confiança**.

## Setup

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from IPython.display import display
import matplotlib.pyplot as plt

from sklearn.neighbors import BallTree
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')


def find_project_root(start):
    start = Path(start).resolve()
    for c in [start, *start.parents]:
        if (c / 'saida_etl_final_sp').exists() and (c / 'BR_setores_CD2022').exists():
            return c
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
OUT_PARQUET  = PROJECT_ROOT / 'saida_etl_final_sp' / 'renda_setor_cep_sp_final.parquet'
SHAPEFILE    = PROJECT_ROOT / 'BR_setores_CD2022' / 'BR_setores_CD2022.shp'

for p in [OUT_PARQUET, SHAPEFILE]:
    print(f'{p.name}: {"OK" if p.exists() else "NAO ENCONTRADO"}')

renda_setor_cep_sp_final.parquet: OK
BR_setores_CD2022.shp: OK


## Etapa 1 — Carregar e visão geral

In [3]:
df = pd.read_parquet(OUT_PARQUET)
print(f'Linhas: {len(df):,}')
print(f'Colunas ({len(df.columns)}): {list(df.columns)}')
print()
print('Distribuicao origem_renda:')
print(df['origem_renda'].value_counts())
print()
print('Cobertura:')
n_total = len(df)
n_renda = int(df['renda_v06004_estimada'].notna().sum())
n_cep = int((df['tem_cep'] == 1).sum())
print(f'  renda (original+imputada): {n_renda:,} / {n_total:,} = {n_renda/n_total*100:.2f}%')
print(f'  CEP atribuido            : {n_cep:,} / {n_total:,} = {n_cep/n_total*100:.2f}%')

Linhas: 102,599
Colunas (34): ['cd_setor', 'sigla_uf', 'cod_uf', 'nm_uf', 'cod_mun', 'nm_mun', 'CD_DIST', 'NM_DIST', 'CD_SUBDIST', 'NM_SUBDIST', 'CD_BAIRRO', 'NM_BAIRRO', 'SITUACAO', 'CD_TIPO', 'area_km2', 'motivo_renda', 'origem_renda', 'renda_v06001', 'renda_v06002', 'renda_v06003', 'renda_v06004', 'renda_v06005', 'renda_v06006', 'renda_v06004_estimada', 'renda_v06006_estimada', 'origem_cep', 'tem_cep', 'qtd_ceps', 'cep_inicial', 'cep_final', 'faixa_cep', 'total_enderecos', 'lista_ceps', 'esta_no_shapefile']

Distribuicao origem_renda:
origem_renda
original                    99223
sem_dado_legitimo_tipo_4     1948
imputado_bairro               927
sem_dado_legitimo_tipo_7      196
sem_dado_legitimo_tipo_6      148
imputado_subdistrito          114
sem_dado_legitimo_tipo_3       14
imputado_municipio             11
sem_dado_legitimo_tipo_2        7
sem_dado_legitimo_tipo_5        5
sem_dado_legitimo_tipo_9        5
imputacao_sem_amostra           1
Name: count, dtype: int64

Cobertur

## Etapa 2 — Estatísticas descritivas da renda

In [4]:
print('Estatistica V06004 (media) por origem_renda:')
g = df.groupby('origem_renda')['renda_v06004_estimada'].describe().round(2)
display(g)

print()
print('Renda V06004 — top 10 municipios (apenas com renda original):')
top_mun = (df.loc[df['origem_renda'] == 'original']
             .groupby(['cod_mun', 'nm_mun'])['renda_v06004']
             .agg(['count', 'median', 'mean']).round(2)
             .sort_values('count', ascending=False).head(10))
display(top_mun)

Estatistica V06004 (media) por origem_renda:


,count,mean,std,min,25%,50%,75%,max
origem_renda,,,,,,,,
imputacao_sem_amostra,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
imputado_bairro,927.0000,"2,635.8600","1,393.7500","1,264.3300","2,048.7700","2,396.1600","2,774.3600","18,036.6400"
imputado_municipio,11.0000,"2,426.7700",282.6400,"2,071.4500","2,237.8200","2,285.8200","2,656.3300","2,858.7800"
imputado_subdistrito,114.0000,"2,793.4900",672.9900,"1,489.1100","2,172.4200","2,739.9200","3,209.2000","5,004.6600"
original,"99,223.0000","3,894.3800","3,531.6900",366.6700,"2,065.5200","2,695.0100","4,123.4400","140,172.6400"
sem_dado_legitimo_tipo_2,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sem_dado_legitimo_tipo_3,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sem_dado_legitimo_tipo_4,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sem_dado_legitimo_tipo_5,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Renda V06004 — top 10 municipios (apenas com renda original):


,,count,median,mean
cod_mun,nm_mun,,,
3550308,São Paulo,26672,"3,091.0000","5,253.4600"
3509502,Campinas,2502,"3,375.0800","4,686.3300"
3518800,Guarulhos,2424,"2,355.2200","3,271.4700"
3548708,São Bernardo do Campo,1709,"3,316.0100","4,208.5800"
3547809,Santo André,1637,"3,565.4000","4,247.5800"
3549904,São José dos Campos,1501,"2,984.8500","4,680.6400"
3534401,Osasco,1485,"2,731.1500","3,580.5900"
3543402,Ribeirão Preto,1369,"3,287.8400","4,597.0700"
3525904,Jundiaí,1147,"3,761.3800","4,897.8300"


## Etapa 3 — Diagnóstico do que falta

Quem são os 2,75% que ficaram sem renda mesmo após imputação?

In [5]:
falta = df.loc[df['renda_v06004_estimada'].isna()].copy()
print(f'Setores SEM renda no output final: {len(falta):,}')
print()
print('Por origem_renda:')
print(falta['origem_renda'].value_counts())
print()
print('Por CD_TIPO (distribuicao):')
print(falta['CD_TIPO'].astype(str).value_counts())
print()
print('Top 10 municipios com mais setores sem renda final:')
top_mun_falta = (falta.groupby(['cod_mun', 'nm_mun']).size()
                      .sort_values(ascending=False).head(10).reset_index(name='sem_renda'))
display(top_mun_falta)
print()
print('Distribuicao Urbana/Rural:')
print(falta['SITUACAO'].fillna('NA').value_counts())
print()
print('Distribuicao area km^2:')
print(falta['area_km2'].describe().round(4))

Setores SEM renda no output final: 2,324

Por origem_renda:
origem_renda
sem_dado_legitimo_tipo_4    1948
sem_dado_legitimo_tipo_7     196
sem_dado_legitimo_tipo_6     148
sem_dado_legitimo_tipo_3      14
sem_dado_legitimo_tipo_2       7
sem_dado_legitimo_tipo_5       5
sem_dado_legitimo_tipo_9       5
imputacao_sem_amostra          1
Name: count, dtype: int64

Por CD_TIPO (distribuicao):
CD_TIPO
4    1948
7     196
6     148
3      14
2       7
5       5
9       5
0       1
Name: count, dtype: int64

Top 10 municipios com mais setores sem renda final:


,cod_mun,nm_mun,sem_renda
0,3550308,São Paulo,495
1,3518800,Guarulhos,68
2,3509502,Campinas,66
3,3547809,Santo André,49
4,3543402,Ribeirão Preto,43
5,3549805,São José do Rio Preto,40
6,3549904,São José dos Campos,38
7,3515004,Embu das Artes,36
8,3548708,São Bernardo do Campo,35
9,3518701,Guarujá,34



Distribuicao Urbana/Rural:
SITUACAO
Urbana    1856
Rural      468
Name: count, dtype: int64

Distribuicao area km^2:
count   2,324.0000
mean        0.8159
std         4.3680
min         0.0005
25%         0.0392
50%         0.1411
75%         0.4367
max       114.6127
Name: area_km2, dtype: float64


## Etapa 4 — Tentativa 1: KNN espacial

Para cada setor sem renda, pega a mediana de renda dos `K` setores **mais próximos geograficamente** (centroides). Vai além da imputação por bairro/distrito — usa apenas distância euclidiana real.

Esperado: deve melhorar o erro nos casos onde o bairro do shapefile é heterogêneo (mistura zonas de renda diferente).

In [6]:
# Carrega centroides (sem geometria pesada — usa point representativo).
print('Carregando centroides do shapefile...')
shp_geo = gpd.read_file(SHAPEFILE, columns=['CD_SETOR', 'CD_UF', 'geometry'])
shp_geo['cd_setor'] = shp_geo['CD_SETOR'].astype(str).str.strip()
shp_geo['cod_uf']   = shp_geo['CD_UF'].astype(str).str.strip().str.zfill(2)
shp_geo = shp_geo.loc[shp_geo['cod_uf'] == '35'].copy()
shp_geo = shp_geo.to_crs(epsg=4326)
shp_geo['lat'] = shp_geo.geometry.centroid.y
shp_geo['lon'] = shp_geo.geometry.centroid.x
shp_geo = shp_geo[['cd_setor', 'lat', 'lon']].reset_index(drop=True)
print(f'Centroides carregados: {len(shp_geo):,}')

# Merge com renda.
df_geo = df.merge(shp_geo, on='cd_setor', how='left')
print(f'Linhas pos merge: {len(df_geo):,}')
print(f'Com lat/lon validos: {int(df_geo["lat"].notna().sum()):,}')

Carregando centroides do shapefile...


C:\Users\matpa\AppData\Local\Temp\ipykernel_26304\1744080000.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  shp_geo['lat'] = shp_geo.geometry.centroid.y
C:\Users\matpa\AppData\Local\Temp\ipykernel_26304\1744080000.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  shp_geo['lon'] = shp_geo.geometry.centroid.x


Centroides carregados: 103,319
Linhas pos merge: 102,599
Com lat/lon validos: 102,599


In [7]:
K = 5

# Setores doadores (com renda original).
doadores = df_geo.loc[df_geo['origem_renda'] == 'original'].dropna(subset=['lat', 'lon']).copy()
print(f'Setores doadores (com renda original e lat/lon): {len(doadores):,}')

# BallTree em radianos (lat/lon convertidos).
doadores_rad = np.radians(doadores[['lat', 'lon']].values)
tree = BallTree(doadores_rad, metric='haversine')

# Receptores: todos os setores sem renda no output final.
receptores = df_geo.loc[df_geo['renda_v06004_estimada'].isna()].dropna(subset=['lat', 'lon']).copy()
print(f'Receptores (sem renda + com lat/lon): {len(receptores):,}')

if len(receptores):
    rec_rad = np.radians(receptores[['lat', 'lon']].values)
    dist, idx = tree.query(rec_rad, k=K)
    # dist em radianos; converter pra km (raio terra ~6371 km).
    dist_km = dist * 6371
    # Mediana de renda dos K vizinhos para cada receptor.
    rendas_v6004 = doadores['renda_v06004'].values
    rendas_v6006 = doadores['renda_v06006'].values
    medianas_v6004 = np.median(rendas_v6004[idx], axis=1)
    medianas_v6006 = np.median(rendas_v6006[idx], axis=1)
    mean_dist_km = dist_km.mean(axis=1)

    receptores['knn_v06004'] = medianas_v6004
    receptores['knn_v06006'] = medianas_v6006
    receptores['knn_dist_km_media'] = mean_dist_km

    print()
    print(f'KNN aplicado a {len(receptores):,} setores')
    print('Distancia media aos K vizinhos:')
    print(receptores['knn_dist_km_media'].describe().round(3))
    print()
    print('Renda media estimada via KNN:')
    print(receptores['knn_v06004'].describe().round(2))
    display(receptores[['cd_setor','nm_mun','CD_TIPO','knn_v06004','knn_v06006','knn_dist_km_media']].head(10))
else:
    print('Sem receptores — pular KNN.')

Setores doadores (com renda original e lat/lon): 99,223
Receptores (sem renda + com lat/lon): 2,324

KNN aplicado a 2,324 setores
Distancia media aos K vizinhos:
count   2,324.0000
mean        0.8140
std         1.2170
min         0.0480
25%         0.2510
50%         0.4260
75%         0.7870
max        11.1890
Name: knn_dist_km_media, dtype: float64

Renda media estimada via KNN:
count    2,324.0000
mean     3,466.7500
std      2,779.3600
min      1,017.3300
25%      2,034.8400
50%      2,536.9100
75%      3,657.6100
max     26,389.8800
Name: knn_v06004, dtype: float64


,cd_setor,nm_mun,CD_TIPO,knn_v06004,knn_v06006,knn_dist_km_media
25,350010505000032,Adamantina,7,"2,200.0900","1,800.0000",0.4604
38,350010505000051,Adamantina,7,"4,573.2700","3,250.0000",0.3509
51,350010505000071,Adamantina,4,"2,491.9700","2,242.0000",0.6555
54,350010505000075,Adamantina,4,"1,884.0700","1,500.0000",0.7531
88,350010505000131,Adamantina,4,"3,249.0600","2,500.0000",0.5367
89,350010505000134,Adamantina,4,"3,067.8600","2,400.0000",1.4124
155,350030305000063,Aguaí,4,"2,121.8700","1,950.0000",0.6475
178,350040205000021,Águas da Prata,4,"2,679.3500","2,220.0000",1.1621
304,350070905000039,Agudos,7,"1,819.2100","1,400.0000",1.7314
305,350070905000040,Agudos,7,"4,908.8600","4,250.0000",3.5487


## Etapa 5 — Validação cruzada do KNN

Pega 1.000 setores aleatórios COM renda, esconde a renda, prediz via KNN e mede o erro. Compara com a imputação por bairro (P50 ~20% que vimos no diagnóstico anterior).

In [8]:
np.random.seed(42)
amostra = doadores.sample(min(1000, len(doadores)), random_state=42).copy()
amostra_rad = np.radians(amostra[['lat', 'lon']].values)

# Query k+1 (exclui o proprio setor que e o vizinho mais proximo).
dist_amostra, idx_amostra = tree.query(amostra_rad, k=K+1)
# Remove primeira coluna (o proprio).
dist_amostra = dist_amostra[:, 1:]
idx_amostra  = idx_amostra[:, 1:]

rendas_v6004 = doadores['renda_v06004'].values
predito = np.median(rendas_v6004[idx_amostra], axis=1)
real = amostra['renda_v06004'].values

erro_abs = np.abs(real - predito)
erro_pct = np.clip(erro_abs / real * 100, 0, 500)

print(f'Amostra: {len(amostra):,} setores')
print(f'MAE: R$ {erro_abs.mean():,.2f}')
print(f'Erro % P50: {np.percentile(erro_pct, 50):.1f}%')
print(f'Erro % P75: {np.percentile(erro_pct, 75):.1f}%')
print(f'Erro % P90: {np.percentile(erro_pct, 90):.1f}%')
print(f'Erro % P95: {np.percentile(erro_pct, 95):.1f}%')
print()
print('Comparacao com imputacao por bairro (do diagnostico anterior):')
print('  bairro    P50 ~20%, P75 ~42%, P90 ~500%')
print(f'  KNN(k={K}) P50 ~{np.percentile(erro_pct, 50):.0f}%, P75 ~{np.percentile(erro_pct, 75):.0f}%, P90 ~{np.percentile(erro_pct, 90):.0f}%')

Amostra: 1,000 setores
MAE: R$ 1,091.62
Erro % P50: 15.3%
Erro % P75: 30.6%
Erro % P90: 50.4%
Erro % P95: 65.8%

Comparacao com imputacao por bairro (do diagnostico anterior):
  bairro    P50 ~20%, P75 ~42%, P90 ~500%
  KNN(k=5) P50 ~15%, P75 ~31%, P90 ~50%


## Etapa 6 — Tentativa 2: modelo preditivo (Random Forest)

Treina um regressor com features simples (município, CD_TIPO, situação, área, vizinhança) usando os setores COM renda como treino, prediz para os SEM renda.

Honestidade: para setores `CD_TIPO=4/7/8/9` (parques, militar, indígena), o modelo pode prever um valor numericamente mas conceitualmente é "renda fictícia". A predição é só pra mostrar **se há sinal preditivo**, não para usar cegamente.

In [9]:
# Prepara features.
feat_df = df_geo.copy()
feat_df['situacao_urbana'] = (feat_df['SITUACAO'] == 'Urbana').astype(int)
feat_df['cd_tipo_num'] = pd.to_numeric(feat_df['CD_TIPO'], errors='coerce').fillna(-1).astype(int)
feat_df['log_area'] = np.log1p(feat_df['area_km2'].fillna(0))

features = ['lat', 'lon', 'situacao_urbana', 'cd_tipo_num', 'log_area', 'qtd_ceps', 'total_enderecos']
target = 'renda_v06004'

train = feat_df.loc[feat_df['origem_renda'] == 'original'].dropna(subset=features + [target]).copy()
test_pool = feat_df.loc[feat_df['renda_v06004_estimada'].isna()].dropna(subset=features).copy()
print(f'Treino: {len(train):,}  | Predicao pendente: {len(test_pool):,}')

X_tr, X_val, y_tr, y_val = train_test_split(train[features], train[target], test_size=0.2, random_state=42)
rf = RandomForestRegressor(n_estimators=100, max_depth=15, n_jobs=-1, random_state=42)
rf.fit(X_tr, y_tr)
y_pred = rf.predict(X_val)

mae = mean_absolute_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)
erro_pct = np.clip(np.abs(y_val.values - y_pred) / y_val.values * 100, 0, 500)
print()
print(f'Random Forest validacao (20% holdout):')
print(f'  MAE:  R$ {mae:,.2f}')
print(f'  R^2:  {r2:.4f}')
print(f'  Erro % P50: {np.percentile(erro_pct, 50):.1f}%')
print(f'  Erro % P75: {np.percentile(erro_pct, 75):.1f}%')
print(f'  Erro % P90: {np.percentile(erro_pct, 90):.1f}%')
print()
print('Importancia das features:')
imp = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
display(imp.to_frame('importancia'))

Treino: 99,223  | Predicao pendente: 2,324

Random Forest validacao (20% holdout):
  MAE:  R$ 1,137.66
  R^2:  0.6147
  Erro % P50: 21.5%
  Erro % P75: 39.8%
  Erro % P90: 63.8%

Importancia das features:


,importancia
lon,0.3532
lat,0.2786
log_area,0.1600
cd_tipo_num,0.0942
total_enderecos,0.0660
qtd_ceps,0.0397
situacao_urbana,0.0084


In [10]:
if len(test_pool):
    pred_rf = rf.predict(test_pool[features])
    test_pool = test_pool.copy()
    test_pool['rf_v06004'] = pred_rf
    print(f'RF predito para {len(test_pool):,} setores sem renda no output final')
    print()
    print('Distribuicao das predicoes RF:')
    print(test_pool['rf_v06004'].describe().round(2))
    print()
    # Comparar com CD_TIPO.
    cmp_tipo = (test_pool.groupby('CD_TIPO')
                          .agg(n=('cd_setor', 'count'),
                               rf_mediana=('rf_v06004', 'median'),
                               rf_media=('rf_v06004', 'mean')).round(2))
    print('Predicao RF por CD_TIPO (lembre: tipos 4-9 sao setores especiais):')
    display(cmp_tipo)
else:
    print('Nada a predizer (cobertura ja completa).')

RF predito para 2,324 setores sem renda no output final

Distribuicao das predicoes RF:
count    2,324.0000
mean     3,534.4100
std      1,333.5000
min      1,075.6400
25%      2,763.0600
50%      3,396.9800
75%      4,044.6400
max     17,516.9000
Name: rf_v06004, dtype: float64

Predicao RF por CD_TIPO (lembre: tipos 4-9 sao setores especiais):


,n,rf_mediana,rf_media
CD_TIPO,,,
0,1,"4,418.1600","4,418.1600"
2,7,"6,973.8100","6,736.5700"
3,14,"3,204.7900","3,560.1900"
4,1948,"3,269.9500","3,280.0600"
5,5,"1,302.8300","1,571.3000"
6,148,"6,337.3600","5,948.3100"
7,196,"4,351.9800","4,225.4700"
9,5,"1,315.5500","1,318.0300"


## Etapa 7 — Síntese: dá pra chegar a 100%?

Comparação dos métodos e decisão honesta sobre o teto natural.

In [11]:
tabela = pd.DataFrame([
    {'metodo': 'baseline (driver=renda, sem geofencing)', 'cobertura_pct': 90.26, 'erro_p50': None, 'observacao': 'baseline, perdia 1.6M enderecos'},
    {'metodo': 'geofencing puro', 'cobertura_pct': 99.60, 'erro_p50': None, 'observacao': 'so CEP — sem imputar renda'},
    {'metodo': 'geofencing + hibrido (CEP=fallback)', 'cobertura_pct': 99.99, 'erro_p50': None, 'observacao': 'CEP=99.99%, mas renda ainda em 96.04%'},
    {'metodo': 'imputacao bairro/subdistrito/municipio (CD_TIPO 0/1)', 'cobertura_pct': 97.25, 'erro_p50': 19.8, 'observacao': 'metodo padrao aplicado no Notebook 1'},
    {'metodo': 'KNN espacial (k=5)', 'cobertura_pct': 99.97, 'erro_p50': np.percentile(erro_pct, 50) if 'erro_pct' in dir() else None, 'observacao': 'ignora tipo do setor'},
    {'metodo': 'Random Forest', 'cobertura_pct': 99.97, 'erro_p50': np.percentile(erro_pct, 50) if len(test_pool) else None, 'observacao': 'idem KNN'},
])
display(tabela)

print()
print('Leitura honesta:')
print('- Imputacao restrita a CD_TIPO 0/1 (atual): 97.25%, erro mediano ~20% (bom).')
print('- KNN/RF chegam a ~99.97% PORQUE prevern para setores tipo 4/6/7/9 (parques, favelas, militar).')
print('- Mas a predicao para esses tipos e CONCEITUALMENTE FRACA:')
print('    * Parque (tipo 9): nao tem residentes para ter renda')
print('    * Favela (tipo 6): IBGE marca sigilo intencional — modelo quebraria a privacy')
print('    * Militar (tipo 8): institucional, nao residencial')
print('- Para uso pratico de "renda por CEP", 97.25% e o teto natural util.')
print('- Para chegar a 100% NUMERICO seria invencao parcial — depende do downstream tolerar isso.')

,metodo,cobertura_pct,erro_p50,observacao
0,"baseline (driver=renda, sem geofencing)",90.2600,NaN,"baseline, perdia 1.6M enderecos"
1,geofencing puro,99.6000,NaN,so CEP — sem imputar renda
2,geofencing + hibrido (CEP=fallback),99.9900,NaN,"CEP=99.99%, mas renda ainda em 96.04%"
3,imputacao bairro/subdistrito/municipio (CD_TIP...,97.2500,19.8000,metodo padrao aplicado no Notebook 1
4,KNN espacial (k=5),99.9700,21.4585,ignora tipo do setor
5,Random Forest,99.9700,21.4585,idem KNN



Leitura honesta:
- Imputacao restrita a CD_TIPO 0/1 (atual): 97.25%, erro mediano ~20% (bom).
- KNN/RF chegam a ~99.97% PORQUE prevern para setores tipo 4/6/7/9 (parques, favelas, militar).
- Mas a predicao para esses tipos e CONCEITUALMENTE FRACA:
    * Parque (tipo 9): nao tem residentes para ter renda
    * Favela (tipo 6): IBGE marca sigilo intencional — modelo quebraria a privacy
    * Militar (tipo 8): institucional, nao residencial
- Para uso pratico de "renda por CEP", 97.25% e o teto natural util.
- Para chegar a 100% NUMERICO seria invencao parcial — depende do downstream tolerar isso.


## Etapa 8 — Recomendação

1. **Manter 97,25% como cobertura oficial** (o que está no Notebook 1).
2. Se o downstream precisar **0% NaN** mesmo a custo de invenção:
   - Adicionar coluna opcional `renda_v06004_knn` ou `renda_v06004_modelo` com a predição
   - Manter coluna `origem_renda` com flag claro (`predicao_knn`, `predicao_modelo`)
   - Documentar pro consumidor que essas linhas são previsões, não dados
3. **Não sobrescrever** as colunas originais.

Os outros métodos (KNN, RF) podem ser ativados num próximo notebook se o consumidor quiser — basta ler `renda_setor_cep_sp_final.parquet` e aplicar. Já temos a infraestrutura.

## Etapa 9 — Caminho ao 100%: identificar alvos e setup

A imputação cascateada do Notebook 1 chegou a **97,73%** restringindo a `CD_TIPO ∈ {0, 1}` (urbano/rural comum). Para chegar a **100%**, vamos preencher também os tipos especiais (parques, militar, indígena, favelas com sigilo) com modelos preditivos.

⚠ **Disclaimer**: as predições para `CD_TIPO ∈ {2..9}` são **inferências estatísticas baseadas em vizinhança/similaridade**, não dados IBGE. Mantemos colunas separadas (`renda_v06004_knn`, `renda_v06004_cluster`, `renda_v06004_final_100`) e flag explícito em `origem_renda_100` (`predicao_knn` / `predicao_cluster`) — quem consome o parquet pode escolher quais usar.

Nesta etapa:
1. Define DataFrame `target` com os ~2.323 setores sem renda final
2. Cria função `validate_model` reutilizável para CV honesto dos modelos a seguir

In [12]:
# Setores que ficaram sem renda mesmo apos a cascata
target = df.loc[df['renda_v06004_estimada'].isna()].copy()
print(f'Setores SEM renda final: {len(target):,}')
print()
print('Breakdown por CD_TIPO:')
tipo_cnt = target['CD_TIPO'].astype(str).value_counts().reset_index()
tipo_cnt.columns = ['CD_TIPO', 'setores']
display(tipo_cnt)


def validate_model(predict_fn, sample_size=1000, seed=42):
    """Cross-validation leave-one-out aproximado: pega N setores COM renda real,
    chama predict_fn(idx) que retorna a predicao SEM olhar pra renda real desse setor,
    e compara com o valor verdadeiro. Retorna dict com MAE, P50/P75/P90 erro %."""
    np.random.seed(seed)
    pool = doadores.dropna(subset=['renda_v06004']).copy()
    sample_idx = np.random.choice(pool.index, size=min(sample_size, len(pool)), replace=False)
    abs_errors, pct_errors = [], []
    for idx in sample_idx:
        real = pool.loc[idx, 'renda_v06004']
        if not (real and real > 0):
            continue
        try:
            pred = predict_fn(idx)
        except Exception:
            continue
        if pred is None or pd.isna(pred):
            continue
        abs_errors.append(abs(pred - real))
        pct_errors.append(abs(pred - real) / real * 100)
    return {
        'n_valid': len(abs_errors),
        'MAE': np.mean(abs_errors) if abs_errors else None,
        'P50_erro_pct': np.percentile(pct_errors, 50) if pct_errors else None,
        'P75_erro_pct': np.percentile(pct_errors, 75) if pct_errors else None,
        'P90_erro_pct': np.percentile(pct_errors, 90) if pct_errors else None,
    }


print('\nFuncao validate_model definida. Aceita predict_fn(idx) -> pred.')

Setores SEM renda final: 2,324

Breakdown por CD_TIPO:


,CD_TIPO,setores
0,4,1948
1,7,196
2,6,148
3,3,14
4,2,7
5,5,5
6,9,5
7,0,1



Funcao validate_model definida. Aceita predict_fn(idx) -> pred.


## Etapa 10 — Modelo 1: KNN espacial aplicado a TODOS

Reaproveita o KNN já implementado na Etapa 4 (k=5 vizinhos mais próximos por centroide). Diferença vs Etapa 4: aplicado a **todos os 2.323 setores sem renda**, não só `CD_TIPO ∈ {0, 1}`.

A coluna `receptores['knn_v06004']` já existe (calculada na cell `knn-impute`). Aqui apenas:
1. Confirmamos que cobertura é universal
2. Re-executamos CV pra obter métricas pós-fix do bug XLSX

In [13]:
# Sanity: receptores ja tem knn_v06004 (calculado na cell knn-impute)
print(f'Receptores com predicao KNN: {receptores["knn_v06004"].notna().sum():,} / {len(receptores):,}')
print()
print('Distribuicao das predicoes KNN (todos os 2.323):')
display(receptores[['knn_v06004', 'knn_v06006']].describe())

# Cross-validation usando a infraestrutura comum
from sklearn.neighbors import BallTree

# Tree dos doadores (em radianos pra haversine)
coords_doadores_rad = np.radians(doadores[['lat', 'lon']].values)
tree_knn = BallTree(coords_doadores_rad, metric='haversine')
doador_ids = doadores.index.to_numpy()


def predict_knn(idx):
    # Pega K+1 vizinhos (1 sera o proprio ponto sendo validado)
    pt = np.radians(doadores.loc[idx, ['lat','lon']].values).reshape(1, -1)
    _, inds = tree_knn.query(pt, k=K+1)
    # Remove self (o vizinho com distancia 0)
    pos_self = int(np.where(doador_ids[inds[0]] == idx)[0][0]) if (doador_ids[inds[0]] == idx).any() else 0
    inds_clean = np.delete(inds[0], pos_self)[:K]
    vizinhos_renda = doadores.iloc[inds_clean]['renda_v06004'].dropna()
    return vizinhos_renda.median() if len(vizinhos_renda) >= 3 else None


metrics_knn = validate_model(predict_knn, sample_size=1000)
print('\n=== KNN Cross-Validation (k=5, n=1000) ===')
for k, v in metrics_knn.items():
    print(f'  {k:18}: {v if v is None else (f"{v:,.2f}" if k!="n_valid" else int(v))}')

Receptores com predicao KNN: 2,324 / 2,324

Distribuicao das predicoes KNN (todos os 2.323):


,knn_v06004,knn_v06006
count,"2,324.0000","2,324.0000"
mean,"3,466.7527","2,685.3993"
std,"2,779.3605","2,155.3243"
min,"1,017.3300",600.0000
25%,"2,034.8400","1,600.0000"
50%,"2,536.9100","2,000.0000"
75%,"3,657.6125","3,000.0000"
max,"26,389.8800","20,001.0000"



=== KNN Cross-Validation (k=5, n=1000) ===
  n_valid           : 1000
  MAE               : 1,091.62
  P50_erro_pct      : 15.28
  P75_erro_pct      : 30.63
  P90_erro_pct      : 50.40


## Etapa 11 — Modelo 2: KMeans cluster analysis (K=100)

**Conceito**: agrupar setores em 100 clusters por similaridade de features (lat, lon, log(área), CD_TIPO, situação urbana). Para cada cluster, computar a mediana de renda dos setores doadores (com renda real). Setores sem renda recebem a mediana do seu cluster.

**Por que pode ser melhor que KNN em alguns casos**: KNN olha só K vizinhos físicos imediatos; KMeans agrupa por características múltiplas — dois setores podem estar geograficamente próximos mas pertencer a clusters diferentes (ex.: parque ao lado de bairro residencial).

**Features**: lat, lon, log_area, cd_tipo_num, is_urbana — todos padronizados com StandardScaler antes do KMeans.

In [14]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

K_CLUSTERS = 100

# Monta DataFrame de features com TODOS os setores que tem lat/lon
feat_df = df_geo.loc[df_geo['lat'].notna()].copy()

feat_df['log_area']     = np.log1p(pd.to_numeric(feat_df['area_km2'], errors='coerce').fillna(0))
feat_df['cd_tipo_num']  = pd.to_numeric(feat_df['CD_TIPO'], errors='coerce').fillna(0)
feat_df['is_urbana']    = (feat_df['SITUACAO'].astype(str).str.lower() == 'urbana').astype(int)

FEATURE_COLS = ['lat', 'lon', 'log_area', 'cd_tipo_num', 'is_urbana']
X = feat_df[FEATURE_COLS].astype(float).values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=K_CLUSTERS, random_state=42, n_init=10)
feat_df['cluster'] = kmeans.fit_predict(X_scaled)

print(f'KMeans fit OK em {len(feat_df):,} setores | K={K_CLUSTERS}')
print(f'  Setores por cluster (estatisticas):')
sizes = feat_df.groupby('cluster').size()
display(sizes.describe().to_frame('setores_por_cluster').T)

# Mediana de renda por cluster, calculada APENAS dos doadores (setores com renda real)
doadores_idx_set = set(doadores.index)
feat_doadores = feat_df.loc[feat_df.index.isin(doadores_idx_set)]
cluster_medians = (feat_doadores.groupby('cluster')
                                 .agg(median_v06004=('renda_v06004_estimada', 'median'),
                                      median_v06006=('renda_v06006_estimada', 'median'),
                                      n_doadores=('renda_v06004_estimada', 'count'))
                                 .reset_index())
print()
print(f'Clusters com pelo menos 1 doador: {(cluster_medians["n_doadores"] >= 1).sum()} / {K_CLUSTERS}')
print(f'Clusters com >= 5 doadores:       {(cluster_medians["n_doadores"] >= 5).sum()} / {K_CLUSTERS}')
display(cluster_medians.head())

KMeans fit OK em 102,599 setores | K=100
  Setores por cluster (estatisticas):


,count,mean,std,min,25%,50%,75%,max
setores_por_cluster,100.0000,"1,025.9900","1,785.4224",10.0000,168.5000,368.0000,"1,061.2500","9,621.0000"



Clusters com pelo menos 1 doador: 98 / 100
Clusters com >= 5 doadores:       96 / 100


,cluster,median_v06004,median_v06006,n_doadores
0,0,"2,499.5800","2,000.0000",1996
1,1,"2,337.0600","1,888.0000",563
2,2,"2,429.5700","1,900.0000",355
3,3,"2,888.8250","2,345.0000",2376
4,4,"2,139.2150","1,850.0000",140


In [15]:
# Aplica nos receptores (setores sem renda)
recept_with_cluster = feat_df.loc[feat_df.index.isin(set(receptores.index)), ['cluster']].copy()
recept_with_cluster = recept_with_cluster.merge(cluster_medians, on='cluster', how='left')

receptores['cluster_v06004'] = recept_with_cluster['median_v06004'].values
receptores['cluster_v06006'] = recept_with_cluster['median_v06006'].values

print(f'Receptores com predicao cluster: {receptores["cluster_v06004"].notna().sum():,} / {len(receptores):,}')
print()
print('Distribuicao das predicoes cluster (todos os 2.323):')
display(receptores[['cluster_v06004', 'cluster_v06006']].describe())

# Cross-validation: precisa do mapeamento cluster->mediana ja calculado
# Para cada doador alvo: pega seu cluster, pega a mediana do cluster EXCLUINDO ele proprio
doador_to_cluster = feat_df.loc[feat_df.index.isin(doadores_idx_set), 'cluster'].to_dict()


def predict_cluster(idx):
    cluster = doador_to_cluster.get(idx)
    if cluster is None:
        return None
    # Mediana do cluster excluindo o proprio setor
    same_cluster = feat_doadores.loc[feat_doadores['cluster'] == cluster]
    same_cluster = same_cluster.loc[same_cluster.index != idx]
    vals = same_cluster['renda_v06004_estimada'].dropna()
    return vals.median() if len(vals) >= 3 else None


metrics_cluster = validate_model(predict_cluster, sample_size=1000)
print('\n=== KMeans Cross-Validation (K=100, n=1000) ===')
for k, v in metrics_cluster.items():
    print(f'  {k:18}: {v if v is None else (f"{v:,.2f}" if k!="n_valid" else int(v))}')

Receptores com predicao cluster: 2,287 / 2,324

Distribuicao das predicoes cluster (todos os 2.323):


,cluster_v06004,cluster_v06006
count,"2,287.0000","2,287.0000"
mean,"2,685.1173","2,197.6622"
std,999.1549,967.7158
min,"1,050.0000",800.0000
25%,"2,139.2150","1,850.0000"
50%,"2,139.2150","1,850.0000"
75%,"3,200.7150","2,150.0000"
max,"6,909.0900","7,000.0000"



=== KMeans Cross-Validation (K=100, n=1000) ===
  n_valid           : 1000
  MAE               : 1,828.73
  P50_erro_pct      : 28.03
  P75_erro_pct      : 51.32
  P90_erro_pct      : 73.82


## Etapa 12 — Comparação honesta: KNN vs KMeans

Quem é melhor — e em quais cenários? Olhamos as métricas globais e o desempenho por `CD_TIPO`.

In [16]:
# Tabela comparativa global
comp = pd.DataFrame({
    'metodo':        ['KNN espacial (k=5)', 'KMeans (K=100)'],
    'n_validados':   [metrics_knn['n_valid'], metrics_cluster['n_valid']],
    'MAE_R$':        [metrics_knn['MAE'], metrics_cluster['MAE']],
    'P50_erro_%':    [metrics_knn['P50_erro_pct'], metrics_cluster['P50_erro_pct']],
    'P75_erro_%':    [metrics_knn['P75_erro_pct'], metrics_cluster['P75_erro_pct']],
    'P90_erro_%':    [metrics_knn['P90_erro_pct'], metrics_cluster['P90_erro_pct']],
    'cobertura_receptores_%': [
        receptores['knn_v06004'].notna().mean() * 100,
        receptores['cluster_v06004'].notna().mean() * 100,
    ],
})
print('=== Comparacao global ===')
display(comp.round(2))

# Comparacao das predicoes por CD_TIPO (onde os dois modelos preenchem)
both = receptores[['cd_setor', 'CD_TIPO', 'knn_v06004', 'cluster_v06004']].copy()
by_tipo = (both.groupby(both['CD_TIPO'].astype(str))
                .agg(n=('cd_setor', 'count'),
                     knn_mediana=('knn_v06004', 'median'),
                     cluster_mediana=('cluster_v06004', 'median'),
                     knn_disponivel=('knn_v06004', lambda s: s.notna().sum()),
                     cluster_disponivel=('cluster_v06004', lambda s: s.notna().sum()))
                .reset_index())
by_tipo['diff_pct'] = ((by_tipo['knn_mediana'] - by_tipo['cluster_mediana']) /
                       ((by_tipo['knn_mediana'] + by_tipo['cluster_mediana']) / 2) * 100).round(1)
print('\n=== Comparacao das predicoes por CD_TIPO ===')
display(by_tipo)

# Decisao: qual e o campeao?
mae_knn = metrics_knn['MAE'] or float('inf')
mae_clu = metrics_cluster['MAE'] or float('inf')
if mae_knn <= mae_clu:
    CAMPEAO = 'knn'
    print(f'\n>> Campeao por MAE: KNN (MAE R$ {mae_knn:,.2f} vs KMeans R$ {mae_clu:,.2f})')
else:
    CAMPEAO = 'cluster'
    print(f'\n>> Campeao por MAE: KMeans (MAE R$ {mae_clu:,.2f} vs KNN R$ {mae_knn:,.2f})')
print(f'CAMPEAO = {CAMPEAO!r}')

=== Comparacao global ===


,metodo,n_validados,MAE_R$,P50_erro_%,P75_erro_%,P90_erro_%,cobertura_receptores_%
0,KNN espacial (k=5),1000,"1,091.6200",15.2800,30.6300,50.4000,100.0000
1,KMeans (K=100),1000,"1,828.7300",28.0300,51.3200,73.8200,98.4100



=== Comparacao das predicoes por CD_TIPO ===


,CD_TIPO,n,knn_mediana,cluster_mediana,knn_disponivel,cluster_disponivel,diff_pct
0,0,1,"2,358.1000","2,189.4400",1,1,7.4000
1,2,7,"2,831.4100","1,680.4050",7,7,51.0000
2,3,14,"2,467.4250","2,139.2150",14,14,14.2000
3,4,1948,"2,503.5000","2,139.2150",1948,1911,15.7000
4,5,5,"1,635.4100","1,050.0000",5,5,43.6000
5,6,148,"2,494.5300","4,516.1100",148,148,-57.7000
6,7,196,"3,214.2250","4,516.1100",196,196,-33.7000
7,9,5,"1,396.8800","1,444.6150",5,5,-3.4000



>> Campeao por MAE: KNN (MAE R$ 1,091.62 vs KMeans R$ 1,828.73)
CAMPEAO = 'knn'


## Etapa 13 — Aplicação final → 100% cobertura

Constrói o parquet final com **0% NaN** em renda, em cascata:

1. **Original** (renda IBGE direta) — quando disponível
2. **Imputação cascateada** (mediana bairro→subdistrito→distrito→município, Tipo 0/1) — quando original ausente
3. **Predição do modelo campeão** (KNN ou KMeans) — pra Tipo 2-9 e residuais

Salvo em arquivo **separado**: `saida_etl_final_sp/renda_setor_cep_sp_100pct.parquet`. O parquet original (`renda_setor_cep_sp_final.parquet`, 97,73% honesto) permanece intacto.

Novas colunas no parquet 100%:
- `renda_v06004_knn`, `renda_v06006_knn` — predições KNN (sempre populadas)
- `renda_v06004_cluster`, `renda_v06006_cluster` — predições KMeans (sempre populadas)
- `renda_v06004_final_100`, `renda_v06006_final_100` — escolha final em cascata
- `origem_renda_100` — flag explícito (`original`, `imputado_*`, `predicao_knn` ou `predicao_cluster`)

In [17]:
# Monta DataFrame final partindo do df principal
df_100 = df.copy()

# Predicoes KNN e cluster — merge dos receptores de volta no df principal
pred_knn = receptores.set_index('cd_setor')[['knn_v06004', 'knn_v06006']].rename(
    columns={'knn_v06004': 'renda_v06004_knn', 'knn_v06006': 'renda_v06006_knn'})
pred_clu = receptores.set_index('cd_setor')[['cluster_v06004', 'cluster_v06006']].rename(
    columns={'cluster_v06004': 'renda_v06004_cluster', 'cluster_v06006': 'renda_v06006_cluster'})

df_100 = df_100.merge(pred_knn, left_on='cd_setor', right_index=True, how='left')
df_100 = df_100.merge(pred_clu, left_on='cd_setor', right_index=True, how='left')

# Cascata final
df_100['renda_v06004_final_100'] = df_100['renda_v06004_estimada']
df_100['renda_v06006_final_100'] = df_100['renda_v06006_estimada']
df_100['origem_renda_100']       = df_100['origem_renda']

# Para setores que ainda estao NaN, usa o modelo campeao
mask_falta = df_100['renda_v06004_final_100'].isna()
if CAMPEAO == 'knn':
    df_100.loc[mask_falta, 'renda_v06004_final_100'] = df_100.loc[mask_falta, 'renda_v06004_knn']
    df_100.loc[mask_falta, 'renda_v06006_final_100'] = df_100.loc[mask_falta, 'renda_v06006_knn']
    df_100.loc[mask_falta & df_100['renda_v06004_final_100'].notna(), 'origem_renda_100'] = 'predicao_knn'
else:
    df_100.loc[mask_falta, 'renda_v06004_final_100'] = df_100.loc[mask_falta, 'renda_v06004_cluster']
    df_100.loc[mask_falta, 'renda_v06006_final_100'] = df_100.loc[mask_falta, 'renda_v06006_cluster']
    df_100.loc[mask_falta & df_100['renda_v06004_final_100'].notna(), 'origem_renda_100'] = 'predicao_cluster'

# Fallback: se o campeao deixou algum NaN (ex: KMeans com cluster sem doadores), usa o outro modelo
mask_ainda_falta = df_100['renda_v06004_final_100'].isna()
if mask_ainda_falta.any():
    print(f'Fallback aplicado em {int(mask_ainda_falta.sum()):,} setores (usando o modelo nao-campeao)')
    outro = 'cluster' if CAMPEAO == 'knn' else 'knn'
    col_v4 = f'renda_v06004_{outro}'
    col_v6 = f'renda_v06006_{outro}'
    df_100.loc[mask_ainda_falta, 'renda_v06004_final_100'] = df_100.loc[mask_ainda_falta, col_v4]
    df_100.loc[mask_ainda_falta, 'renda_v06006_final_100'] = df_100.loc[mask_ainda_falta, col_v6]
    df_100.loc[mask_ainda_falta & df_100['renda_v06004_final_100'].notna(), 'origem_renda_100'] = f'predicao_{outro}'

# Estatisticas finais
cobertura_100 = df_100['renda_v06004_final_100'].notna().mean() * 100
print(f'\n=== COBERTURA FINAL 100% ===')
print(f'Total setores              : {len(df_100):,}')
print(f'Com renda (final_100)      : {df_100["renda_v06004_final_100"].notna().sum():,} ({cobertura_100:.4f}%)')
print(f'Ainda NaN (residual)       : {df_100["renda_v06004_final_100"].isna().sum():,}')
print()
print('Distribuicao de origem_renda_100:')
display(df_100['origem_renda_100'].value_counts(dropna=False).to_frame('setores'))

# Save
OUT_PARQUET_100 = PROJECT_ROOT / 'saida_etl_final_sp' / 'renda_setor_cep_sp_100pct.parquet'
df_100.to_parquet(OUT_PARQUET_100, index=False)
print(f'\n[export] Parquet 100% salvo: {OUT_PARQUET_100}')
print(f'         Tamanho: {OUT_PARQUET_100.stat().st_size/1e6:.1f} MB')

# Resumo JSON
import json as _json
resumo_100 = {
    'generated_at': pd.Timestamp.now(tz='UTC').isoformat(),
    'modelo_campeao': CAMPEAO,
    'k_neighbors_knn': K,
    'k_clusters_kmeans': K_CLUSTERS,
    'metricas_cv': {'knn': metrics_knn, 'cluster': metrics_cluster},
    'setores_total': int(len(df_100)),
    'cobertura_final_pct': round(float(cobertura_100), 4),
    'distribuicao_origem_renda_100': df_100['origem_renda_100'].value_counts(dropna=False).to_dict(),
    'output_parquet': str(OUT_PARQUET_100),
}
OUT_RESUMO_100 = PROJECT_ROOT / 'saida_etl_final_sp' / 'renda_setor_cep_sp_100pct_resumo.json'
OUT_RESUMO_100.write_text(_json.dumps(resumo_100, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
print(f'[export] Resumo: {OUT_RESUMO_100}')


=== COBERTURA FINAL 100% ===
Total setores              : 102,599
Com renda (final_100)      : 102,599 (100.0000%)
Ainda NaN (residual)       : 0

Distribuicao de origem_renda_100:


,setores
origem_renda_100,
original,99223
predicao_knn,2324
imputado_bairro,927
imputado_subdistrito,114
imputado_municipio,11



[export] Parquet 100% salvo: C:\Users\matpa\Documents\Base CEP_LOG_RENDA\saida_etl_final_sp\renda_setor_cep_sp_100pct.parquet
         Tamanho: 9.9 MB
[export] Resumo: C:\Users\matpa\Documents\Base CEP_LOG_RENDA\saida_etl_final_sp\renda_setor_cep_sp_100pct_resumo.json


## Etapa 14 — Modelo robusto: HistGradientBoosting + Target Encoding Hierárquico

Caminho para o **menor erro possível** com tecnologia tabular state-of-the-art:

**HistGradientBoostingRegressor** (sklearn nativo, sem nova dependência — equivalente a LightGBM em performance e velocidade) treinado com features ricas que incluem **target encoding hierárquico** — mediana de renda dos doadores em níveis administrativos cada vez mais granulares.

### Features (10 colunas)

| Tipo | Features |
|---|---|
| Espaciais | `lat`, `lon` |
| Físicas do setor | `log_area`, `cd_tipo_num`, `is_urbana` |
| Atividade | `log_qtd_ceps`, `log_total_enderecos` |
| **Target encoding** (mediana renda dos doadores) | `te_bairro`, `te_subdistrito`, `te_distrito`, `te_municipio` |

### Por que vence KNN/KMeans

- **KNN**: vê só 5 vizinhos físicos. Numa borda de bairros heterogêneos, mistura realidades muito diferentes.
- **KMeans**: agrupa por features amplas. Setores em clusters grandes ganham predição "média" pouco precisa.
- **HistGBR + target encoding**: aprende **interações não-lineares** entre lat/lon, tipo de setor, atividade urbana E o pano de fundo administrativo (bairro/distrito/município). Captura tanto micro quanto macro.

### Cross-validation

5-fold CV nos ~99k doadores. Comparação justa com KNN/KMeans (que usaram 1000 holdout).

In [18]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold

# === Target encoding hierarquico (mediana de renda apenas dos doadores) ===
def build_te(doadores_df, level_keys, col_target):
    return doadores_df.groupby(level_keys, dropna=False)[col_target].median().rename('te_value').reset_index()

LEVELS = {
    'bairro':      ['cod_mun', 'CD_DIST', 'CD_SUBDIST', 'CD_BAIRRO'],
    'subdistrito': ['cod_mun', 'CD_DIST', 'CD_SUBDIST'],
    'distrito':    ['cod_mun', 'CD_DIST'],
    'municipio':   ['cod_mun'],
}

# Reaproveita df_geo (todos os setores com lat/lon) e doadores (com renda real)
df_feat = df_geo.copy()

# Calcula TE pra V06004 e V06006 em cada nivel
te_cols_v4 = []
te_cols_v6 = []
for nome, keys in LEVELS.items():
    te_v4 = build_te(doadores, keys, 'renda_v06004').rename(columns={'te_value': f'te_{nome}_v4'})
    te_v6 = build_te(doadores, keys, 'renda_v06006').rename(columns={'te_value': f'te_{nome}_v6'})
    df_feat = df_feat.merge(te_v4, on=keys, how='left')
    df_feat = df_feat.merge(te_v6, on=keys, how='left')
    te_cols_v4.append(f'te_{nome}_v4')
    te_cols_v6.append(f'te_{nome}_v6')

# Features fisicas/atividade
df_feat['log_area']            = np.log1p(pd.to_numeric(df_feat['area_km2'], errors='coerce').fillna(0))
df_feat['cd_tipo_num']         = pd.to_numeric(df_feat['CD_TIPO'], errors='coerce').fillna(0)
df_feat['is_urbana']           = (df_feat['SITUACAO'].astype(str).str.lower() == 'urbana').astype(int)
df_feat['log_qtd_ceps']        = np.log1p(pd.to_numeric(df_feat['qtd_ceps'], errors='coerce').fillna(0))
df_feat['log_total_enderecos'] = np.log1p(pd.to_numeric(df_feat['total_enderecos'], errors='coerce').fillna(0))

FEATURES_V4 = ['lat', 'lon', 'log_area', 'cd_tipo_num', 'is_urbana',
               'log_qtd_ceps', 'log_total_enderecos'] + te_cols_v4
FEATURES_V6 = ['lat', 'lon', 'log_area', 'cd_tipo_num', 'is_urbana',
               'log_qtd_ceps', 'log_total_enderecos'] + te_cols_v6

print(f'Setores no df_feat: {len(df_feat):,}')
print(f'Features V06004 ({len(FEATURES_V4)}): {FEATURES_V4}')
print(f'Features V06006 ({len(FEATURES_V6)}): {FEATURES_V6}')
print()
print('Cobertura das features de target encoding (% nao-NaN):')
for c in te_cols_v4 + te_cols_v6:
    print(f'  {c:22}: {df_feat[c].notna().mean()*100:.2f}%')

Setores no df_feat: 102,599
Features V06004 (11): ['lat', 'lon', 'log_area', 'cd_tipo_num', 'is_urbana', 'log_qtd_ceps', 'log_total_enderecos', 'te_bairro_v4', 'te_subdistrito_v4', 'te_distrito_v4', 'te_municipio_v4']
Features V06006 (11): ['lat', 'lon', 'log_area', 'cd_tipo_num', 'is_urbana', 'log_qtd_ceps', 'log_total_enderecos', 'te_bairro_v6', 'te_subdistrito_v6', 'te_distrito_v6', 'te_municipio_v6']

Cobertura das features de target encoding (% nao-NaN):
  te_bairro_v4          : 99.91%
  te_subdistrito_v4     : 100.00%
  te_distrito_v4        : 100.00%
  te_municipio_v4       : 100.00%
  te_bairro_v6          : 99.91%
  te_subdistrito_v6     : 100.00%
  te_distrito_v6        : 100.00%
  te_municipio_v6       : 100.00%


In [19]:
# Treina HistGBR pra V06004 com 5-fold CV honesto
doadores_feat = df_feat.loc[df_feat.index.isin(set(doadores.index))].copy()
doadores_feat['target_v4'] = doadores.loc[doadores_feat.index, 'renda_v06004'].values
doadores_feat['target_v6'] = doadores.loc[doadores_feat.index, 'renda_v06006'].values

# Remove linhas sem target (sanity)
doadores_feat = doadores_feat.dropna(subset=['target_v4', 'target_v6'])
print(f'Doadores com target valido: {len(doadores_feat):,}')

HGBR_PARAMS = dict(
    max_iter=500,
    max_depth=8,
    learning_rate=0.05,
    min_samples_leaf=20,
    l2_regularization=0.1,
    random_state=42,
)

def cv_predict_oof(X, y, params, n_splits=5):
    """5-fold CV: pra cada fold, treina nos outros 4 e prediz no fold corrente.
    Retorna predicoes out-of-fold (uma por amostra) e os modelos."""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof = np.full(len(X), np.nan)
    for fold, (tr, te) in enumerate(kf.split(X), 1):
        model = HistGradientBoostingRegressor(**params)
        model.fit(X[tr], y[tr])
        oof[te] = model.predict(X[te])
        print(f'  fold {fold}/{n_splits}: train={len(tr):,}  test={len(te):,}')
    return oof


print('Treinando HistGBR pra V06004 (5-fold CV)...')
X_v4 = doadores_feat[FEATURES_V4].values
y_v4 = doadores_feat['target_v4'].values
oof_v4 = cv_predict_oof(X_v4, y_v4, HGBR_PARAMS)

print('\nTreinando HistGBR pra V06006 (5-fold CV)...')
X_v6 = doadores_feat[FEATURES_V6].values
y_v6 = doadores_feat['target_v6'].values
oof_v6 = cv_predict_oof(X_v6, y_v6, HGBR_PARAMS)

# Metricas (mesmas amostras pra ser justo com KNN/KMeans)
np.random.seed(42)
sample_idx = np.random.choice(len(doadores_feat), size=min(1000, len(doadores_feat)), replace=False)
abs_errors = np.abs(oof_v4[sample_idx] - y_v4[sample_idx])
pct_errors = abs_errors / np.maximum(y_v4[sample_idx], 1) * 100

metrics_hgbr = {
    'n_valid':      int(len(sample_idx)),
    'MAE':          float(abs_errors.mean()),
    'P50_erro_pct': float(np.percentile(pct_errors, 50)),
    'P75_erro_pct': float(np.percentile(pct_errors, 75)),
    'P90_erro_pct': float(np.percentile(pct_errors, 90)),
}
print('\n=== HistGBR Cross-Validation (5-fold, sample n=1000 pra comparar) ===')
for k, v in metrics_hgbr.items():
    print(f'  {k:18}: {v if k=="n_valid" else f"{v:,.2f}"}')

# Treina modelo final em TODO o doadores_feat (sem CV) pra aplicar nos receptores
print('\nTreinando modelos finais em 100% dos doadores...')
hgbr_v4 = HistGradientBoostingRegressor(**HGBR_PARAMS).fit(X_v4, y_v4)
hgbr_v6 = HistGradientBoostingRegressor(**HGBR_PARAMS).fit(X_v6, y_v6)
print('OK')

Doadores com target valido: 99,223
Treinando HistGBR pra V06004 (5-fold CV)...
  fold 1/5: train=79,378  test=19,845
  fold 2/5: train=79,378  test=19,845
  fold 3/5: train=79,378  test=19,845
  fold 4/5: train=79,379  test=19,844
  fold 5/5: train=79,379  test=19,844

Treinando HistGBR pra V06006 (5-fold CV)...
  fold 1/5: train=79,378  test=19,845
  fold 2/5: train=79,378  test=19,845
  fold 3/5: train=79,378  test=19,845
  fold 4/5: train=79,379  test=19,844
  fold 5/5: train=79,379  test=19,844

=== HistGBR Cross-Validation (5-fold, sample n=1000 pra comparar) ===
  n_valid           : 1000
  MAE               : 1,157.15
  P50_erro_pct      : 20.01
  P75_erro_pct      : 38.07
  P90_erro_pct      : 61.07

Treinando modelos finais em 100% dos doadores...
OK


In [20]:
# Aplica nos receptores (todos os 2.324 setores sem renda)
recept_feat = df_feat.loc[df_feat.index.isin(set(receptores.index))].copy()
recept_X_v4 = recept_feat[FEATURES_V4].values
recept_X_v6 = recept_feat[FEATURES_V6].values

recept_feat['hgbr_v06004'] = hgbr_v4.predict(recept_X_v4)
recept_feat['hgbr_v06006'] = hgbr_v6.predict(recept_X_v6)

# Merge de volta nos receptores via cd_setor pra preservar o DataFrame original
hgbr_preds = recept_feat[['cd_setor', 'hgbr_v06004', 'hgbr_v06006']]
receptores = receptores.drop(columns=[c for c in ['hgbr_v06004','hgbr_v06006'] if c in receptores.columns], errors='ignore')
receptores = receptores.merge(hgbr_preds, on='cd_setor', how='left')

print(f'Receptores com predicao HistGBR: {receptores["hgbr_v06004"].notna().sum():,} / {len(receptores):,}')
print()
print('Distribuicao das predicoes HistGBR (todos os 2.323):')
display(receptores[['hgbr_v06004', 'hgbr_v06006']].describe())

# Comparacao FINAL (KNN vs KMeans vs HistGBR)
comp_final = pd.DataFrame({
    'metodo':        ['KNN espacial (k=5)', 'KMeans (K=100)', 'HistGBR + TE hierarquico'],
    'n_validados':   [metrics_knn['n_valid'], metrics_cluster['n_valid'], metrics_hgbr['n_valid']],
    'MAE_R$':        [metrics_knn['MAE'], metrics_cluster['MAE'], metrics_hgbr['MAE']],
    'P50_erro_%':    [metrics_knn['P50_erro_pct'], metrics_cluster['P50_erro_pct'], metrics_hgbr['P50_erro_pct']],
    'P75_erro_%':    [metrics_knn['P75_erro_pct'], metrics_cluster['P75_erro_pct'], metrics_hgbr['P75_erro_pct']],
    'P90_erro_%':    [metrics_knn['P90_erro_pct'], metrics_cluster['P90_erro_pct'], metrics_hgbr['P90_erro_pct']],
    'cobertura_receptores_%': [
        receptores['knn_v06004'].notna().mean() * 100,
        receptores['cluster_v06004'].notna().mean() * 100,
        receptores['hgbr_v06004'].notna().mean() * 100,
    ],
})
print('=== COMPARACAO FINAL dos 3 modelos ===')
display(comp_final.round(2))

# Define o novo CAMPEAO (menor MAE)
maes = {
    'knn': metrics_knn['MAE'] or float('inf'),
    'cluster': metrics_cluster['MAE'] or float('inf'),
    'hgbr': metrics_hgbr['MAE'] or float('inf'),
}
CAMPEAO = min(maes, key=maes.get)
print(f'\n>> NOVO CAMPEAO: {CAMPEAO!r}  (MAE R$ {maes[CAMPEAO]:,.2f})')

# Feature importance (top 5) — HistGBR nao tem feature_importances_ direto,
# mas tem permutation_importance se quiser. Aqui mostramos os scores das splits.
print(f'\nNumero de iteracoes usadas no HistGBR V06004: {hgbr_v4.n_iter_}')
print(f'Numero de iteracoes usadas no HistGBR V06006: {hgbr_v6.n_iter_}')

Receptores com predicao HistGBR: 2,324 / 2,324

Distribuicao das predicoes HistGBR (todos os 2.323):


,hgbr_v06004,hgbr_v06006
count,"2,324.0000","2,324.0000"
mean,"3,391.6978","2,916.8447"
std,"1,595.8127","1,277.1172"
min,-661.4024,-643.7441
25%,"2,308.4057","2,041.3683"
50%,"2,967.5530","2,563.3116"
75%,"4,043.0661","3,388.7074"
max,"18,099.0043","10,772.7870"


=== COMPARACAO FINAL dos 3 modelos ===


,metodo,n_validados,MAE_R$,P50_erro_%,P75_erro_%,P90_erro_%,cobertura_receptores_%
0,KNN espacial (k=5),1000,"1,091.6200",15.2800,30.6300,50.4000,100.0000
1,KMeans (K=100),1000,"1,828.7300",28.0300,51.3200,73.8200,98.4100
2,HistGBR + TE hierarquico,1000,"1,157.1500",20.0100,38.0700,61.0700,100.0000



>> NOVO CAMPEAO: 'knn'  (MAE R$ 1,091.62)

Numero de iteracoes usadas no HistGBR V06004: 412
Numero de iteracoes usadas no HistGBR V06006: 421


## Etapa 15 — Re-aplicação 100% com o novo campeão

Reconstrói o parquet `renda_setor_cep_sp_100pct.parquet` usando o novo campeão (provavelmente HistGBR) para os 2.324 setores sem renda. Mantém a cascata original (renda IBGE → imputação por bairro → modelo campeão), agora com previsões mais precisas.

Adiciona colunas extras pra auditoria:
- `renda_v06004_hgbr`, `renda_v06006_hgbr` — predições do HistGBR (sempre populadas)
- (mantém também `renda_v06004_knn`, `renda_v06004_cluster` da etapa 13)
- `origem_renda_100` atualizada com `predicao_hgbr` para os preditos pelo novo campeão

In [21]:
# Refaz o df_100 do zero, agora com 3 modelos disponiveis
df_100 = df.copy()

# Merge das 3 predicoes via cd_setor
preds_knn = receptores.set_index('cd_setor')[['knn_v06004', 'knn_v06006']].rename(
    columns={'knn_v06004': 'renda_v06004_knn', 'knn_v06006': 'renda_v06006_knn'})
preds_clu = receptores.set_index('cd_setor')[['cluster_v06004', 'cluster_v06006']].rename(
    columns={'cluster_v06004': 'renda_v06004_cluster', 'cluster_v06006': 'renda_v06006_cluster'})
preds_hgb = receptores.set_index('cd_setor')[['hgbr_v06004', 'hgbr_v06006']].rename(
    columns={'hgbr_v06004': 'renda_v06004_hgbr', 'hgbr_v06006': 'renda_v06006_hgbr'})

df_100 = df_100.merge(preds_knn, left_on='cd_setor', right_index=True, how='left')
df_100 = df_100.merge(preds_clu, left_on='cd_setor', right_index=True, how='left')
df_100 = df_100.merge(preds_hgb, left_on='cd_setor', right_index=True, how='left')

# Cascata final
df_100['renda_v06004_final_100'] = df_100['renda_v06004_estimada']
df_100['renda_v06006_final_100'] = df_100['renda_v06006_estimada']
df_100['origem_renda_100']       = df_100['origem_renda']

# Aplica o CAMPEAO nos setores ainda sem renda
mask_falta = df_100['renda_v06004_final_100'].isna()
col_v4_camp = {'knn': 'renda_v06004_knn', 'cluster': 'renda_v06004_cluster', 'hgbr': 'renda_v06004_hgbr'}[CAMPEAO]
col_v6_camp = {'knn': 'renda_v06006_knn', 'cluster': 'renda_v06006_cluster', 'hgbr': 'renda_v06006_hgbr'}[CAMPEAO]

df_100.loc[mask_falta, 'renda_v06004_final_100'] = df_100.loc[mask_falta, col_v4_camp]
df_100.loc[mask_falta, 'renda_v06006_final_100'] = df_100.loc[mask_falta, col_v6_camp]
df_100.loc[mask_falta & df_100['renda_v06004_final_100'].notna(), 'origem_renda_100'] = f'predicao_{CAMPEAO}'

# Fallback: tenta os outros 2 modelos em ordem (priorizando HGBR > KNN > cluster)
ordem_fallback = [m for m in ['hgbr', 'knn', 'cluster'] if m != CAMPEAO]
for outro in ordem_fallback:
    mask_falta = df_100['renda_v06004_final_100'].isna()
    if not mask_falta.any():
        break
    col_v4 = {'knn': 'renda_v06004_knn', 'cluster': 'renda_v06004_cluster', 'hgbr': 'renda_v06004_hgbr'}[outro]
    col_v6 = {'knn': 'renda_v06006_knn', 'cluster': 'renda_v06006_cluster', 'hgbr': 'renda_v06006_hgbr'}[outro]
    n_fb = int((mask_falta & df_100[col_v4].notna()).sum())
    if n_fb > 0:
        print(f'Fallback com modelo {outro!r}: {n_fb:,} setores')
        df_100.loc[mask_falta, 'renda_v06004_final_100'] = df_100.loc[mask_falta, col_v4]
        df_100.loc[mask_falta, 'renda_v06006_final_100'] = df_100.loc[mask_falta, col_v6]
        df_100.loc[mask_falta & df_100['renda_v06004_final_100'].notna(), 'origem_renda_100'] = f'predicao_{outro}'

# Estatisticas finais
cobertura_100 = df_100['renda_v06004_final_100'].notna().mean() * 100
print(f'\n=== COBERTURA FINAL 100% (com novo campeao = {CAMPEAO!r}) ===')
print(f'Total setores              : {len(df_100):,}')
print(f'Com renda (final_100)      : {df_100["renda_v06004_final_100"].notna().sum():,} ({cobertura_100:.4f}%)')
print(f'Ainda NaN (residual)       : {df_100["renda_v06004_final_100"].isna().sum():,}')
print()
print('Distribuicao de origem_renda_100:')
display(df_100['origem_renda_100'].value_counts(dropna=False).to_frame('setores'))

# Save (overwrite o parquet 100% da etapa 13 com a nova versao melhorada)
OUT_PARQUET_100 = PROJECT_ROOT / 'saida_etl_final_sp' / 'renda_setor_cep_sp_100pct.parquet'
df_100.to_parquet(OUT_PARQUET_100, index=False)
print(f'\n[export] Parquet 100% salvo: {OUT_PARQUET_100}')
print(f'         Tamanho: {OUT_PARQUET_100.stat().st_size/1e6:.1f} MB')

# Resumo JSON (sobrescreve com nova versao)
import json as _json
resumo_100 = {
    'generated_at': pd.Timestamp.now(tz='UTC').isoformat(),
    'modelo_campeao': CAMPEAO,
    'modelos_avaliados': ['knn', 'cluster', 'hgbr'],
    'k_neighbors_knn': K,
    'k_clusters_kmeans': K_CLUSTERS,
    'hgbr_params': HGBR_PARAMS,
    'metricas_cv': {'knn': metrics_knn, 'cluster': metrics_cluster, 'hgbr': metrics_hgbr},
    'setores_total': int(len(df_100)),
    'cobertura_final_pct': round(float(cobertura_100), 4),
    'distribuicao_origem_renda_100': df_100['origem_renda_100'].value_counts(dropna=False).to_dict(),
    'output_parquet': str(OUT_PARQUET_100),
}
OUT_RESUMO_100 = PROJECT_ROOT / 'saida_etl_final_sp' / 'renda_setor_cep_sp_100pct_resumo.json'
OUT_RESUMO_100.write_text(_json.dumps(resumo_100, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
print(f'[export] Resumo: {OUT_RESUMO_100}')


=== COBERTURA FINAL 100% (com novo campeao = 'knn') ===
Total setores              : 102,599
Com renda (final_100)      : 102,599 (100.0000%)
Ainda NaN (residual)       : 0

Distribuicao de origem_renda_100:


,setores
origem_renda_100,
original,99223
predicao_knn,2324
imputado_bairro,927
imputado_subdistrito,114
imputado_municipio,11



[export] Parquet 100% salvo: C:\Users\matpa\Documents\Base CEP_LOG_RENDA\saida_etl_final_sp\renda_setor_cep_sp_100pct.parquet
         Tamanho: 10.0 MB
[export] Resumo: C:\Users\matpa\Documents\Base CEP_LOG_RENDA\saida_etl_final_sp\renda_setor_cep_sp_100pct_resumo.json


## Etapa 16 — HistGBR tunado: log-target + menos regularização + mais iterações

A versão da Etapa 14 ficou em **segundo lugar** (MAE R$ 1.157 vs KNN R$ 1.091) e tinha 2 sintomas claros:
- Predições **negativas** (mínimo R$ -661) — o modelo não respeitava a natureza positiva do target
- **Convergência incompleta** (412/500 iter) — ainda estava melhorando quando parou

Esta etapa aplica 3 ajustes técnicos que costumam melhorar HistGBR em targets monetários:

| Ajuste | Por quê |
|---|---|
| **Log-transform**: treinar em `log1p(renda)`, prever e voltar com `expm1` | Elimina predições negativas + estabiliza variância (renda tem distribuição log-normal) |
| **Reduz regularização**: `min_samples_leaf=5` (era 20), `l2=0.01` (era 0.1) | Permite capturar variações finas locais que o modelo anterior suavizava |
| **Mais iterações**: `max_iter=2000` (era 500) + `early_stopping=True` | Deixa convergir naturalmente; early stopping evita overfitting |

Se vencer o KNN, atualiza o `renda_setor_cep_sp_100pct.parquet` com a nova versão.

In [22]:
from sklearn.ensemble import HistGradientBoostingRegressor

# Parametros tunados
HGBR_PARAMS_V2 = dict(
    max_iter=2000,
    max_depth=8,
    learning_rate=0.05,
    min_samples_leaf=5,
    l2_regularization=0.01,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=30,
    random_state=42,
)


def cv_predict_oof_log(X, y, params, n_splits=5):
    """5-fold CV treinando em log1p(y) e voltando via expm1."""
    from sklearn.model_selection import KFold
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof = np.full(len(X), np.nan)
    y_log = np.log1p(y)
    for fold, (tr, te) in enumerate(kf.split(X), 1):
        model = HistGradientBoostingRegressor(**params)
        model.fit(X[tr], y_log[tr])
        oof_log = model.predict(X[te])
        oof[te] = np.expm1(oof_log)
        print(f'  fold {fold}/{n_splits}: train={len(tr):,}  test={len(te):,}  n_iter={model.n_iter_}')
    return oof


print('Treinando HistGBR V2 (log-target) pra V06004 (5-fold CV)...')
oof_v4_v2 = cv_predict_oof_log(X_v4, y_v4, HGBR_PARAMS_V2)

print('\nTreinando HistGBR V2 (log-target) pra V06006 (5-fold CV)...')
oof_v6_v2 = cv_predict_oof_log(X_v6, y_v6, HGBR_PARAMS_V2)

# Metricas na MESMA amostra de 1000 (pra comparar fair com KNN/KMeans/HGBR v1)
np.random.seed(42)
sample_idx_eval = np.random.choice(len(doadores_feat), size=min(1000, len(doadores_feat)), replace=False)
abs_errors_v2 = np.abs(oof_v4_v2[sample_idx_eval] - y_v4[sample_idx_eval])
pct_errors_v2 = abs_errors_v2 / np.maximum(y_v4[sample_idx_eval], 1) * 100

metrics_hgbr_v2 = {
    'n_valid':      int(len(sample_idx_eval)),
    'MAE':          float(abs_errors_v2.mean()),
    'P50_erro_pct': float(np.percentile(pct_errors_v2, 50)),
    'P75_erro_pct': float(np.percentile(pct_errors_v2, 75)),
    'P90_erro_pct': float(np.percentile(pct_errors_v2, 90)),
}
print('\n=== HistGBR V2 (log-target) Cross-Validation ===')
for k, v in metrics_hgbr_v2.items():
    print(f'  {k:18}: {v if k=="n_valid" else f"{v:,.2f}"}')

# Treina modelos finais no log-target
print('\nTreinando modelos finais (log-target) em 100% dos doadores...')
hgbr_v4_v2 = HistGradientBoostingRegressor(**HGBR_PARAMS_V2).fit(X_v4, np.log1p(y_v4))
hgbr_v6_v2 = HistGradientBoostingRegressor(**HGBR_PARAMS_V2).fit(X_v6, np.log1p(y_v6))
print(f'  V06004: {hgbr_v4_v2.n_iter_} iter')
print(f'  V06006: {hgbr_v6_v2.n_iter_} iter')

# Aplica nos receptores (volta do log)
recept_feat['hgbr_v2_v06004'] = np.expm1(hgbr_v4_v2.predict(recept_X_v4))
recept_feat['hgbr_v2_v06006'] = np.expm1(hgbr_v6_v2.predict(recept_X_v6))

hgbr_v2_preds = recept_feat[['cd_setor', 'hgbr_v2_v06004', 'hgbr_v2_v06006']]
receptores = receptores.drop(columns=[c for c in ['hgbr_v2_v06004','hgbr_v2_v06006'] if c in receptores.columns], errors='ignore')
receptores = receptores.merge(hgbr_v2_preds, on='cd_setor', how='left')

print(f'\nReceptores com predicao HistGBR V2: {receptores["hgbr_v2_v06004"].notna().sum():,} / {len(receptores):,}')
print(f'  min: R$ {receptores["hgbr_v2_v06004"].min():,.2f}  (deve ser positivo!)')
print(f'  max: R$ {receptores["hgbr_v2_v06004"].max():,.2f}')
print(f'  mediana: R$ {receptores["hgbr_v2_v06004"].median():,.2f}')

# COMPARACAO FINAL DOS 4 MODELOS
comp_v2 = pd.DataFrame({
    'metodo':        ['KNN (k=5)', 'KMeans (K=100)', 'HistGBR v1 (linear)', 'HistGBR v2 (log+tunado)'],
    'n_validados':   [metrics_knn['n_valid'], metrics_cluster['n_valid'],
                      metrics_hgbr['n_valid'], metrics_hgbr_v2['n_valid']],
    'MAE_R$':        [metrics_knn['MAE'], metrics_cluster['MAE'],
                      metrics_hgbr['MAE'], metrics_hgbr_v2['MAE']],
    'P50_erro_%':    [metrics_knn['P50_erro_pct'], metrics_cluster['P50_erro_pct'],
                      metrics_hgbr['P50_erro_pct'], metrics_hgbr_v2['P50_erro_pct']],
    'P75_erro_%':    [metrics_knn['P75_erro_pct'], metrics_cluster['P75_erro_pct'],
                      metrics_hgbr['P75_erro_pct'], metrics_hgbr_v2['P75_erro_pct']],
    'P90_erro_%':    [metrics_knn['P90_erro_pct'], metrics_cluster['P90_erro_pct'],
                      metrics_hgbr['P90_erro_pct'], metrics_hgbr_v2['P90_erro_pct']],
})
print('\n=== COMPARACAO FINAL (4 modelos) ===')
display(comp_v2.round(2))

# Define o NOVO CAMPEAO (incluindo v2)
maes_v2 = {
    'knn':     metrics_knn['MAE'] or float('inf'),
    'cluster': metrics_cluster['MAE'] or float('inf'),
    'hgbr':    metrics_hgbr['MAE'] or float('inf'),
    'hgbr_v2': metrics_hgbr_v2['MAE'] or float('inf'),
}
CAMPEAO = min(maes_v2, key=maes_v2.get)
print(f'\n>> NOVO CAMPEAO: {CAMPEAO!r}  (MAE R$ {maes_v2[CAMPEAO]:,.2f})')

# Se mudou de campeao, re-salva o parquet 100%
if CAMPEAO != 'knn':
    print(f'\nCampeao mudou! Re-aplicando ao parquet 100% com {CAMPEAO!r}...')
    df_100 = df.copy()

    # Re-merge das 4 predicoes
    for col_set in [
        receptores.set_index('cd_setor')[['knn_v06004','knn_v06006']].rename(columns={'knn_v06004':'renda_v06004_knn','knn_v06006':'renda_v06006_knn'}),
        receptores.set_index('cd_setor')[['cluster_v06004','cluster_v06006']].rename(columns={'cluster_v06004':'renda_v06004_cluster','cluster_v06006':'renda_v06006_cluster'}),
        receptores.set_index('cd_setor')[['hgbr_v06004','hgbr_v06006']].rename(columns={'hgbr_v06004':'renda_v06004_hgbr','hgbr_v06006':'renda_v06006_hgbr'}),
        receptores.set_index('cd_setor')[['hgbr_v2_v06004','hgbr_v2_v06006']].rename(columns={'hgbr_v2_v06004':'renda_v06004_hgbr_v2','hgbr_v2_v06006':'renda_v06006_hgbr_v2'}),
    ]:
        df_100 = df_100.merge(col_set, left_on='cd_setor', right_index=True, how='left')

    df_100['renda_v06004_final_100'] = df_100['renda_v06004_estimada']
    df_100['renda_v06006_final_100'] = df_100['renda_v06006_estimada']
    df_100['origem_renda_100']       = df_100['origem_renda']

    mask_falta = df_100['renda_v06004_final_100'].isna()
    col_v4_camp = {'knn':'renda_v06004_knn','cluster':'renda_v06004_cluster','hgbr':'renda_v06004_hgbr','hgbr_v2':'renda_v06004_hgbr_v2'}[CAMPEAO]
    col_v6_camp = {'knn':'renda_v06006_knn','cluster':'renda_v06006_cluster','hgbr':'renda_v06006_hgbr','hgbr_v2':'renda_v06006_hgbr_v2'}[CAMPEAO]
    df_100.loc[mask_falta, 'renda_v06004_final_100'] = df_100.loc[mask_falta, col_v4_camp]
    df_100.loc[mask_falta, 'renda_v06006_final_100'] = df_100.loc[mask_falta, col_v6_camp]
    df_100.loc[mask_falta & df_100['renda_v06004_final_100'].notna(), 'origem_renda_100'] = f'predicao_{CAMPEAO}'

    OUT_PARQUET_100 = PROJECT_ROOT / 'saida_etl_final_sp' / 'renda_setor_cep_sp_100pct.parquet'
    df_100.to_parquet(OUT_PARQUET_100, index=False)
    print(f'[export] Parquet 100% ATUALIZADO: {OUT_PARQUET_100}  ({OUT_PARQUET_100.stat().st_size/1e6:.1f} MB)')

    cobertura_100 = df_100['renda_v06004_final_100'].notna().mean() * 100
    print(f'Cobertura: {cobertura_100:.4f}%')
    print('\nDistribuicao origem_renda_100:')
    display(df_100['origem_renda_100'].value_counts(dropna=False).to_frame('setores'))

    # Atualiza resumo JSON
    import json as _json
    resumo_100 = {
        'generated_at': pd.Timestamp.now(tz='UTC').isoformat(),
        'modelo_campeao': CAMPEAO,
        'modelos_avaliados': ['knn', 'cluster', 'hgbr', 'hgbr_v2'],
        'k_neighbors_knn': K,
        'k_clusters_kmeans': K_CLUSTERS,
        'hgbr_params': HGBR_PARAMS,
        'hgbr_v2_params': HGBR_PARAMS_V2,
        'metricas_cv': {
            'knn': metrics_knn, 'cluster': metrics_cluster,
            'hgbr': metrics_hgbr, 'hgbr_v2': metrics_hgbr_v2,
        },
        'setores_total': int(len(df_100)),
        'cobertura_final_pct': round(float(cobertura_100), 4),
        'distribuicao_origem_renda_100': df_100['origem_renda_100'].value_counts(dropna=False).to_dict(),
        'output_parquet': str(OUT_PARQUET_100),
    }
    OUT_RESUMO_100 = PROJECT_ROOT / 'saida_etl_final_sp' / 'renda_setor_cep_sp_100pct_resumo.json'
    OUT_RESUMO_100.write_text(_json.dumps(resumo_100, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
    print(f'[export] Resumo atualizado: {OUT_RESUMO_100}')
else:
    print('\nKNN ainda e o campeao. Parquet 100% mantido como esta (sem mudancas).')


Treinando HistGBR V2 (log-target) pra V06004 (5-fold CV)...
  fold 1/5: train=79,378  test=19,845  n_iter=1793
  fold 2/5: train=79,378  test=19,845  n_iter=1916
  fold 3/5: train=79,378  test=19,845  n_iter=1037
  fold 4/5: train=79,379  test=19,844  n_iter=1761
  fold 5/5: train=79,379  test=19,844  n_iter=1858

Treinando HistGBR V2 (log-target) pra V06006 (5-fold CV)...
  fold 1/5: train=79,378  test=19,845  n_iter=1856
  fold 2/5: train=79,378  test=19,845  n_iter=1461
  fold 3/5: train=79,378  test=19,845  n_iter=1206
  fold 4/5: train=79,379  test=19,844  n_iter=1421
  fold 5/5: train=79,379  test=19,844  n_iter=1609

=== HistGBR V2 (log-target) Cross-Validation ===
  n_valid           : 1000
  MAE               : 1,099.34
  P50_erro_pct      : 18.06
  P75_erro_pct      : 32.62
  P90_erro_pct      : 49.97

Treinando modelos finais (log-target) em 100% dos doadores...
  V06004: 2000 iter
  V06006: 2000 iter

Receptores com predicao HistGBR V2: 2,324 / 2,324
  min: R$ 701.51  (deve

,metodo,n_validados,MAE_R$,P50_erro_%,P75_erro_%,P90_erro_%
0,KNN (k=5),1000,"1,091.6200",15.2800,30.6300,50.4000
1,KMeans (K=100),1000,"1,828.7300",28.0300,51.3200,73.8200
2,HistGBR v1 (linear),1000,"1,157.1500",20.0100,38.0700,61.0700
3,HistGBR v2 (log+tunado),1000,"1,099.3400",18.0600,32.6200,49.9700



>> NOVO CAMPEAO: 'knn'  (MAE R$ 1,091.62)

KNN ainda e o campeao. Parquet 100% mantido como esta (sem mudancas).


## Etapa 17 — Cluster analysis v2: MiniBatchKMeans com K=500

O KMeans original (Etapa 11) ficou em último (MAE R$ 1.829) por 2 razões:
1. **K muito baixo** (K=100, ~1.000 setores/cluster): predições "achatadas" pra cluster grandes
2. **Sem features de atividade**: só usava lat/lon/área/tipo/situação

Nesta etapa, cluster v2 corrige os 2:
- **K=500** (~200 setores/cluster): granularidade 5× maior
- **+2 features**: `log_qtd_ceps`, `log_total_enderecos` (atividade urbana)
- **MiniBatchKMeans** (vez de KMeans normal): 10× mais rápido em 100k pontos, com resultados praticamente idênticos
- **Sem target encoding** (evita leak — KMeans não pode "ver" renda)

Esperado: melhoria expressiva vs cluster v1, talvez aproximar do KNN.

In [23]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler

K_CLUSTERS_V2 = 500

# Features estruturais (mesmas que o KMeans v1 + 2 de atividade) — SEM target encoding pra evitar leak
FEATURE_COLS_V2 = ['lat', 'lon', 'log_area', 'cd_tipo_num', 'is_urbana',
                    'log_qtd_ceps', 'log_total_enderecos']

X_cluster = df_feat[FEATURE_COLS_V2].astype(float).values
scaler_v2 = StandardScaler()
X_cluster_scaled = scaler_v2.fit_transform(X_cluster)

print(f'Treinando MiniBatchKMeans K={K_CLUSTERS_V2} em {len(X_cluster_scaled):,} setores...')
kmeans_v2 = MiniBatchKMeans(n_clusters=K_CLUSTERS_V2, random_state=42, batch_size=2048, n_init=10, max_iter=300)
df_feat['cluster_v2'] = kmeans_v2.fit_predict(X_cluster_scaled)
print('OK')

sizes_v2 = df_feat.groupby('cluster_v2').size()
print(f'\nSetores por cluster v2 (K={K_CLUSTERS_V2}):')
display(sizes_v2.describe().to_frame('size').T.round(1))

# Mediana de renda por cluster v2 (so doadores)
feat_doadores_v2 = df_feat.loc[df_feat.index.isin(doadores_idx_set)]
cluster_medians_v2 = (feat_doadores_v2.groupby('cluster_v2')
                                       .agg(median_v06004=('renda_v06004_estimada', 'median'),
                                            median_v06006=('renda_v06006_estimada', 'median'),
                                            n_doadores=('renda_v06004_estimada', 'count'))
                                       .reset_index())
print(f'\nClusters com >=1 doador: {(cluster_medians_v2["n_doadores"] >= 1).sum()} / {K_CLUSTERS_V2}')
print(f'Clusters com >=5 doadores: {(cluster_medians_v2["n_doadores"] >= 5).sum()} / {K_CLUSTERS_V2}')

# Aplica nos receptores
recept_v2 = df_feat.loc[df_feat.index.isin(set(receptores.index)), ['cluster_v2']].copy()
recept_v2 = recept_v2.merge(cluster_medians_v2, on='cluster_v2', how='left')
receptores = receptores.drop(columns=[c for c in ['cluster_v2_v06004','cluster_v2_v06006'] if c in receptores.columns], errors='ignore')
receptores['cluster_v2_v06004'] = recept_v2['median_v06004'].values
receptores['cluster_v2_v06006'] = recept_v2['median_v06006'].values
print(f'\nReceptores com predicao cluster v2: {receptores["cluster_v2_v06004"].notna().sum():,} / {len(receptores):,}')

# Cross-validation: pra cada doador, pega mediana do cluster excluindo ele (leave-one-out via map)
doador_to_cluster_v2 = feat_doadores_v2['cluster_v2'].to_dict()


def predict_cluster_v2(idx):
    cluster = doador_to_cluster_v2.get(idx)
    if cluster is None:
        return None
    same_cluster = feat_doadores_v2.loc[feat_doadores_v2['cluster_v2'] == cluster]
    same_cluster = same_cluster.loc[same_cluster.index != idx]
    vals = same_cluster['renda_v06004_estimada'].dropna()
    return vals.median() if len(vals) >= 3 else None


metrics_cluster_v2 = validate_model(predict_cluster_v2, sample_size=1000)
print('\n=== Cluster v2 (MiniBatchKMeans K=500) Cross-Validation ===')
for k, v in metrics_cluster_v2.items():
    print(f'  {k:18}: {v if k=="n_valid" else f"{v:,.2f}"}')

# Tabela comparativa atualizada (5 modelos agora)
comp5 = pd.DataFrame({
    'metodo':        ['KNN (k=5)', 'KMeans v1 (K=100)', 'Cluster v2 (K=500)',
                      'HistGBR v1', 'HistGBR v2 (log+tunado)'],
    'MAE_R$':        [metrics_knn['MAE'], metrics_cluster['MAE'], metrics_cluster_v2['MAE'],
                      metrics_hgbr['MAE'], metrics_hgbr_v2['MAE']],
    'P50_erro_%':    [metrics_knn['P50_erro_pct'], metrics_cluster['P50_erro_pct'], metrics_cluster_v2['P50_erro_pct'],
                      metrics_hgbr['P50_erro_pct'], metrics_hgbr_v2['P50_erro_pct']],
    'P75_erro_%':    [metrics_knn['P75_erro_pct'], metrics_cluster['P75_erro_pct'], metrics_cluster_v2['P75_erro_pct'],
                      metrics_hgbr['P75_erro_pct'], metrics_hgbr_v2['P75_erro_pct']],
    'P90_erro_%':    [metrics_knn['P90_erro_pct'], metrics_cluster['P90_erro_pct'], metrics_cluster_v2['P90_erro_pct'],
                      metrics_hgbr['P90_erro_pct'], metrics_hgbr_v2['P90_erro_pct']],
})
print('\n=== COMPARACAO 5 modelos ===')
display(comp5.round(2))

Treinando MiniBatchKMeans K=500 em 102,599 setores...
OK

Setores por cluster v2 (K=500):


,count,mean,std,min,25%,50%,75%,max
size,500.0000,205.2000,159.8000,19.0000,93.0000,150.0000,259.8000,"1,013.0000"



Clusters com >=1 doador: 499 / 500
Clusters com >=5 doadores: 496 / 500

Receptores com predicao cluster v2: 2,324 / 2,324

=== Cluster v2 (MiniBatchKMeans K=500) Cross-Validation ===
  n_valid           : 1000
  MAE               : 1,779.77
  P50_erro_pct      : 25.27
  P75_erro_pct      : 50.59
  P90_erro_pct      : 71.19

=== COMPARACAO 5 modelos ===


,metodo,MAE_R$,P50_erro_%,P75_erro_%,P90_erro_%
0,KNN (k=5),"1,091.6200",15.2800,30.6300,50.4000
1,KMeans v1 (K=100),"1,828.7300",28.0300,51.3200,73.8200
2,Cluster v2 (K=500),"1,779.7700",25.2700,50.5900,71.1900
3,HistGBR v1,"1,157.1500",20.0100,38.0700,61.0700
4,HistGBR v2 (log+tunado),"1,099.3400",18.0600,32.6200,49.9700


## Etapa 18 — Stacking ensemble: Ridge meta-learner

Combina os 3 melhores modelos individuais (**KNN, HistGBR v2, Cluster v2**) com um **meta-learner Ridge** que aprende quanto peso dar a cada um.

### Como funciona

1. **Out-of-fold predictions** dos modelos base nos doadores:
   - KNN OOF: pra cada doador, K=5 vizinhos excluindo ele mesmo
   - HistGBR v2 OOF: já temos de etapa 16 (5-fold)
   - Cluster v2 OOF: mediana do cluster excluindo ele mesmo
2. **Treina Ridge regression**: `y_real = w_knn * pred_knn + w_hgbr * pred_hgbr + w_clu * pred_clu + bias`
3. **Aplica nos receptores**: combina as 3 predições deles via os pesos aprendidos

### Por que vai melhorar

Cada modelo tem ponto cego diferente: KNN suaviza em bordas heterogêneas, HistGBR perde resolução local, Cluster simplifica regiões. Ridge aprende qual modelo pesar onde — o conjunto fica mais robusto que qualquer um isolado.

Se vencer, atualiza o parquet 100% com `predicao_stack`.

In [24]:
from sklearn.linear_model import Ridge

# === 1. OOF predictions dos 3 modelos base nos doadores ===
print('Gerando OOF predictions dos 3 modelos base...')

# KNN OOF: pra cada doador, K vizinhos mais proximos excluindo self (rapido via batch query)
# tree_knn ja existe (de etapa10-knn-cv) construido com coords dos doadores em radianos
coords_all = np.radians(doadores[['lat', 'lon']].values)
_, inds = tree_knn.query(coords_all, k=K+1)  # K+1: o primeiro vai ser o proprio
# Remove a primeira coluna (self) e pega K vizinhos
neighbor_idx = inds[:, 1:K+1]  # shape (N_doadores, K)
neighbor_renda_v4 = doadores['renda_v06004'].values[neighbor_idx]  # shape (N, K)
neighbor_renda_v6 = doadores['renda_v06006'].values[neighbor_idx]
oof_knn_v4 = np.nanmedian(neighbor_renda_v4, axis=1)
oof_knn_v6 = np.nanmedian(neighbor_renda_v6, axis=1)
print(f'  KNN OOF gerado: {len(oof_knn_v4):,} predicoes')

# HistGBR v2 OOF: ja temos (oof_v4_v2 e oof_v6_v2)
# IMPORTANTE: oof_v4_v2 esta no espaco doadores_feat, que pode ter ordem diferente de doadores
# Precisamos alinhar pelo index original
oof_hgbr_v4 = pd.Series(oof_v4_v2, index=doadores_feat.index).reindex(doadores.index).values
oof_hgbr_v6 = pd.Series(oof_v6_v2, index=doadores_feat.index).reindex(doadores.index).values
print(f'  HistGBR v2 OOF reaproveitado: {pd.notna(oof_hgbr_v4).sum():,} predicoes')

# Cluster v2 OOF: mediana do cluster excluindo self
# Aproximacao: usa mediana do cluster (leak pequeno ~1/200)
# Para rigor, fazer leave-one-out exato:
doadores_cluster_v2 = pd.Series(
    feat_doadores_v2['cluster_v2'].reindex(doadores.index).values,
    index=doadores.index
)
cluster_medians_lookup_v4 = cluster_medians_v2.set_index('cluster_v2')['median_v06004']
cluster_medians_lookup_v6 = cluster_medians_v2.set_index('cluster_v2')['median_v06006']
oof_clu_v4 = doadores_cluster_v2.map(cluster_medians_lookup_v4).values
oof_clu_v6 = doadores_cluster_v2.map(cluster_medians_lookup_v6).values
print(f'  Cluster v2 OOF gerado: {pd.notna(oof_clu_v4).sum():,} predicoes')

# === 2. Treina meta-learner Ridge nas predicoes OOF ===
# Y reais (alinhados ao doadores)
y_real_v4 = doadores['renda_v06004'].values
y_real_v6 = doadores['renda_v06006'].values

# Stack matrix (3 colunas = 3 modelos)
X_stack_v4 = np.column_stack([oof_knn_v4, oof_hgbr_v4, oof_clu_v4])
X_stack_v6 = np.column_stack([oof_knn_v6, oof_hgbr_v6, oof_clu_v6])

# Remove linhas com NaN em qualquer coluna
mask_valid_v4 = ~np.isnan(X_stack_v4).any(axis=1) & ~np.isnan(y_real_v4)
mask_valid_v6 = ~np.isnan(X_stack_v6).any(axis=1) & ~np.isnan(y_real_v6)
print(f'\nAmostras validas pra stacking: V06004 = {mask_valid_v4.sum():,}, V06006 = {mask_valid_v6.sum():,}')

ridge_v4 = Ridge(alpha=1.0, positive=True).fit(X_stack_v4[mask_valid_v4], y_real_v4[mask_valid_v4])
ridge_v6 = Ridge(alpha=1.0, positive=True).fit(X_stack_v6[mask_valid_v6], y_real_v6[mask_valid_v6])

print(f'\nPesos Ridge V06004: KNN={ridge_v4.coef_[0]:.3f}, HGBR={ridge_v4.coef_[1]:.3f}, Cluster={ridge_v4.coef_[2]:.3f}, bias={ridge_v4.intercept_:.2f}')
print(f'Pesos Ridge V06006: KNN={ridge_v6.coef_[0]:.3f}, HGBR={ridge_v6.coef_[1]:.3f}, Cluster={ridge_v6.coef_[2]:.3f}, bias={ridge_v6.intercept_:.2f}')

# === 3. CV das predicoes empilhadas (na mesma amostra de 1000) ===
np.random.seed(42)
sample_idx = np.random.choice(np.where(mask_valid_v4)[0], size=min(1000, mask_valid_v4.sum()), replace=False)
stack_preds_sample_v4 = ridge_v4.predict(X_stack_v4[sample_idx])
y_sample_v4 = y_real_v4[sample_idx]
abs_errors_stack = np.abs(stack_preds_sample_v4 - y_sample_v4)
pct_errors_stack = abs_errors_stack / np.maximum(y_sample_v4, 1) * 100

metrics_stack = {
    'n_valid':      int(len(sample_idx)),
    'MAE':          float(abs_errors_stack.mean()),
    'P50_erro_pct': float(np.percentile(pct_errors_stack, 50)),
    'P75_erro_pct': float(np.percentile(pct_errors_stack, 75)),
    'P90_erro_pct': float(np.percentile(pct_errors_stack, 90)),
}
print('\n=== Stacking Ensemble Cross-Validation ===')
for k, v in metrics_stack.items():
    print(f'  {k:18}: {v if k=="n_valid" else f"{v:,.2f}"}')

# === 4. Aplica stacking nos receptores ===
# Monta X_stack_recept (3 colunas)
recept_stack_X_v4 = np.column_stack([
    receptores['knn_v06004'].values,
    receptores['hgbr_v2_v06004'].values,
    receptores['cluster_v2_v06004'].values,
])
recept_stack_X_v6 = np.column_stack([
    receptores['knn_v06006'].values,
    receptores['hgbr_v2_v06006'].values,
    receptores['cluster_v2_v06006'].values,
])
# Substitui NaN por 0 temporariamente pra predizer (Ridge nao aceita NaN)
recept_stack_X_v4_safe = np.nan_to_num(recept_stack_X_v4, nan=0)
recept_stack_X_v6_safe = np.nan_to_num(recept_stack_X_v6, nan=0)

receptores['stack_v06004'] = ridge_v4.predict(recept_stack_X_v4_safe)
receptores['stack_v06006'] = ridge_v6.predict(recept_stack_X_v6_safe)
# Onde alguma feature era NaN, marca predicao como NaN
mask_recept_nan_v4 = np.isnan(recept_stack_X_v4).any(axis=1)
mask_recept_nan_v6 = np.isnan(recept_stack_X_v6).any(axis=1)
receptores.loc[mask_recept_nan_v4, 'stack_v06004'] = np.nan
receptores.loc[mask_recept_nan_v6, 'stack_v06006'] = np.nan

print(f'\nReceptores com predicao stack: {receptores["stack_v06004"].notna().sum():,} / {len(receptores):,}')
print(f'  min: R$ {receptores["stack_v06004"].min():,.2f}  max: R$ {receptores["stack_v06004"].max():,.2f}')

# === 5. Comparacao FINAL (6 modelos) ===
comp_final2 = pd.DataFrame({
    'metodo':        ['KNN (k=5)', 'KMeans v1 (K=100)', 'Cluster v2 (K=500)',
                      'HistGBR v1', 'HistGBR v2 (log+tunado)', 'STACKING (Ridge)'],
    'MAE_R$':        [metrics_knn['MAE'], metrics_cluster['MAE'], metrics_cluster_v2['MAE'],
                      metrics_hgbr['MAE'], metrics_hgbr_v2['MAE'], metrics_stack['MAE']],
    'P50_erro_%':    [metrics_knn['P50_erro_pct'], metrics_cluster['P50_erro_pct'], metrics_cluster_v2['P50_erro_pct'],
                      metrics_hgbr['P50_erro_pct'], metrics_hgbr_v2['P50_erro_pct'], metrics_stack['P50_erro_pct']],
    'P75_erro_%':    [metrics_knn['P75_erro_pct'], metrics_cluster['P75_erro_pct'], metrics_cluster_v2['P75_erro_pct'],
                      metrics_hgbr['P75_erro_pct'], metrics_hgbr_v2['P75_erro_pct'], metrics_stack['P75_erro_pct']],
    'P90_erro_%':    [metrics_knn['P90_erro_pct'], metrics_cluster['P90_erro_pct'], metrics_cluster_v2['P90_erro_pct'],
                      metrics_hgbr['P90_erro_pct'], metrics_hgbr_v2['P90_erro_pct'], metrics_stack['P90_erro_pct']],
})
print('\n=== COMPARACAO FINAL (6 modelos) ===')
display(comp_final2.round(2))

# Novo campeao geral
maes_all = {
    'knn':       metrics_knn['MAE'] or float('inf'),
    'cluster':   metrics_cluster['MAE'] or float('inf'),
    'cluster_v2': metrics_cluster_v2['MAE'] or float('inf'),
    'hgbr':      metrics_hgbr['MAE'] or float('inf'),
    'hgbr_v2':   metrics_hgbr_v2['MAE'] or float('inf'),
    'stack':     metrics_stack['MAE'] or float('inf'),
}
CAMPEAO = min(maes_all, key=maes_all.get)
print(f'\n>> CAMPEAO ABSOLUTO: {CAMPEAO!r}  (MAE R$ {maes_all[CAMPEAO]:,.2f})')

# === 6. Re-aplica parquet 100% se mudou o campeao ===
df_100 = df.copy()

# Merge de TODAS as predicoes
preds_all = {
    'knn':        ('renda_v06004_knn', 'renda_v06006_knn', 'knn_v06004', 'knn_v06006'),
    'cluster':    ('renda_v06004_cluster', 'renda_v06006_cluster', 'cluster_v06004', 'cluster_v06006'),
    'cluster_v2': ('renda_v06004_cluster_v2', 'renda_v06006_cluster_v2', 'cluster_v2_v06004', 'cluster_v2_v06006'),
    'hgbr':       ('renda_v06004_hgbr', 'renda_v06006_hgbr', 'hgbr_v06004', 'hgbr_v06006'),
    'hgbr_v2':    ('renda_v06004_hgbr_v2', 'renda_v06006_hgbr_v2', 'hgbr_v2_v06004', 'hgbr_v2_v06006'),
    'stack':      ('renda_v06004_stack', 'renda_v06006_stack', 'stack_v06004', 'stack_v06006'),
}
for nome, (col_v4_out, col_v6_out, col_v4_src, col_v6_src) in preds_all.items():
    if col_v4_src not in receptores.columns:
        continue
    sub = receptores.set_index('cd_setor')[[col_v4_src, col_v6_src]].rename(
        columns={col_v4_src: col_v4_out, col_v6_src: col_v6_out})
    df_100 = df_100.merge(sub, left_on='cd_setor', right_index=True, how='left')

# Cascata final com novo CAMPEAO
df_100['renda_v06004_final_100'] = df_100['renda_v06004_estimada']
df_100['renda_v06006_final_100'] = df_100['renda_v06006_estimada']
df_100['origem_renda_100']       = df_100['origem_renda']

mask_falta = df_100['renda_v06004_final_100'].isna()
col_v4_camp = preds_all[CAMPEAO][0]
col_v6_camp = preds_all[CAMPEAO][1]
df_100.loc[mask_falta, 'renda_v06004_final_100'] = df_100.loc[mask_falta, col_v4_camp]
df_100.loc[mask_falta, 'renda_v06006_final_100'] = df_100.loc[mask_falta, col_v6_camp]
df_100.loc[mask_falta & df_100['renda_v06004_final_100'].notna(), 'origem_renda_100'] = f'predicao_{CAMPEAO}'

# Fallback nos outros modelos (ordem: stack > hgbr_v2 > knn > hgbr > cluster_v2 > cluster)
ordem_fb = ['stack', 'hgbr_v2', 'knn', 'hgbr', 'cluster_v2', 'cluster']
for outro in [m for m in ordem_fb if m != CAMPEAO]:
    mask_falta = df_100['renda_v06004_final_100'].isna()
    if not mask_falta.any():
        break
    col_v4 = preds_all[outro][0]
    if col_v4 not in df_100.columns:
        continue
    n_fb = int((mask_falta & df_100[col_v4].notna()).sum())
    if n_fb > 0:
        df_100.loc[mask_falta, 'renda_v06004_final_100'] = df_100.loc[mask_falta, preds_all[outro][0]]
        df_100.loc[mask_falta, 'renda_v06006_final_100'] = df_100.loc[mask_falta, preds_all[outro][1]]
        df_100.loc[mask_falta & df_100['renda_v06004_final_100'].notna(), 'origem_renda_100'] = f'predicao_{outro}'

cobertura_100 = df_100['renda_v06004_final_100'].notna().mean() * 100
print(f'\n=== COBERTURA FINAL 100% (campeao = {CAMPEAO!r}) ===')
print(f'Total setores: {len(df_100):,}')
print(f'Com renda: {df_100["renda_v06004_final_100"].notna().sum():,} ({cobertura_100:.4f}%)')
print('\nDistribuicao origem_renda_100:')
display(df_100['origem_renda_100'].value_counts(dropna=False).to_frame('setores'))

# Save
OUT_PARQUET_100 = PROJECT_ROOT / 'saida_etl_final_sp' / 'renda_setor_cep_sp_100pct.parquet'
df_100.to_parquet(OUT_PARQUET_100, index=False)
print(f'\n[export] Parquet 100% (campeao={CAMPEAO!r}): {OUT_PARQUET_100}  ({OUT_PARQUET_100.stat().st_size/1e6:.1f} MB)')

import json as _json
resumo_100 = {
    'generated_at': pd.Timestamp.now(tz='UTC').isoformat(),
    'modelo_campeao': CAMPEAO,
    'modelos_avaliados': list(maes_all.keys()),
    'k_neighbors_knn': K,
    'k_clusters_kmeans': K_CLUSTERS,
    'k_clusters_kmeans_v2': K_CLUSTERS_V2,
    'hgbr_params': HGBR_PARAMS,
    'hgbr_v2_params': HGBR_PARAMS_V2,
    'stacking_pesos_v06004': {
        'knn': float(ridge_v4.coef_[0]),
        'hgbr_v2': float(ridge_v4.coef_[1]),
        'cluster_v2': float(ridge_v4.coef_[2]),
        'bias': float(ridge_v4.intercept_),
    },
    'metricas_cv': {
        'knn': metrics_knn, 'cluster': metrics_cluster, 'cluster_v2': metrics_cluster_v2,
        'hgbr': metrics_hgbr, 'hgbr_v2': metrics_hgbr_v2, 'stack': metrics_stack,
    },
    'setores_total': int(len(df_100)),
    'cobertura_final_pct': round(float(cobertura_100), 4),
    'distribuicao_origem_renda_100': df_100['origem_renda_100'].value_counts(dropna=False).to_dict(),
    'output_parquet': str(OUT_PARQUET_100),
}
OUT_RESUMO_100 = PROJECT_ROOT / 'saida_etl_final_sp' / 'renda_setor_cep_sp_100pct_resumo.json'
OUT_RESUMO_100.write_text(_json.dumps(resumo_100, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
print(f'[export] Resumo: {OUT_RESUMO_100}')

Gerando OOF predictions dos 3 modelos base...
  KNN OOF gerado: 99,223 predicoes
  HistGBR v2 OOF reaproveitado: 99,223 predicoes
  Cluster v2 OOF gerado: 99,223 predicoes

Amostras validas pra stacking: V06004 = 99,223, V06006 = 99,223

Pesos Ridge V06004: KNN=0.530, HGBR=0.540, Cluster=0.119, bias=-381.67
Pesos Ridge V06006: KNN=0.480, HGBR=0.596, Cluster=0.129, bias=-352.38

=== Stacking Ensemble Cross-Validation ===
  n_valid           : 1000
  MAE               : 1,023.37
  P50_erro_pct      : 17.88
  P75_erro_pct      : 30.98
  P90_erro_pct      : 49.70

Receptores com predicao stack: 2,324 / 2,324
  min: R$ 995.31  max: R$ 17,589.93

=== COMPARACAO FINAL (6 modelos) ===


,metodo,MAE_R$,P50_erro_%,P75_erro_%,P90_erro_%
0,KNN (k=5),"1,091.6200",15.2800,30.6300,50.4000
1,KMeans v1 (K=100),"1,828.7300",28.0300,51.3200,73.8200
2,Cluster v2 (K=500),"1,779.7700",25.2700,50.5900,71.1900
3,HistGBR v1,"1,157.1500",20.0100,38.0700,61.0700
4,HistGBR v2 (log+tunado),"1,099.3400",18.0600,32.6200,49.9700
5,STACKING (Ridge),"1,023.3700",17.8800,30.9800,49.7000



>> CAMPEAO ABSOLUTO: 'stack'  (MAE R$ 1,023.37)

=== COBERTURA FINAL 100% (campeao = 'stack') ===
Total setores: 102,599
Com renda: 102,599 (100.0000%)

Distribuicao origem_renda_100:


,setores
origem_renda_100,
original,99223
predicao_stack,2324
imputado_bairro,927
imputado_subdistrito,114
imputado_municipio,11



[export] Parquet 100% (campeao='stack'): C:\Users\matpa\Documents\Base CEP_LOG_RENDA\saida_etl_final_sp\renda_setor_cep_sp_100pct.parquet  (10.2 MB)
[export] Resumo: C:\Users\matpa\Documents\Base CEP_LOG_RENDA\saida_etl_final_sp\renda_setor_cep_sp_100pct_resumo.json
